# S.O.N.A.R.

## Sourcing Operations & Navigation via Agentic Recruitment

### Send the brief. Find the signal. Start sourcing with confidence.

**A recruiter-controlled Sourcing Readiness Agent that converts a job description, hiring-manager intake notes, and clarification answers into a complete, approved, and measurable sourcing brief.**

> **Know what is clear. Surface what is missing. Launch the search with confidence.**

---

**Track:** Agents for Business
**Author:** Shivangi Kapoor
**Program:** UC Davis MSBA
**Hackathon:** Google × Kaggle AI Agents Intensive Vibe Coding Capstone
**Version:** Prototype v0.2
**Last Updated:** July 2026

---

## Project Overview

Recruiting teams often begin sourcing before a role is fully defined. Important requirements may remain ambiguous, contradictory, or undocumented until candidates have already been sourced, screened, or rejected.

S.O.N.A.R. addresses this problem before sourcing begins.

The agent:

1. reads the job description;
2. analyzes recruiter notes or an intake-call transcript;
3. extracts confirmed role requirements;
4. identifies missing, ambiguous, and contradictory information;
5. calculates a Sourcing Readiness Score;
6. prioritizes the questions that materially affect the search;
7. updates the brief as new answers are provided;
8. requires recruiter approval before sourcing begins;
9. estimates the turnaround time to the first qualified shortlist.

## Current Prototype Scope

**Job description + intake notes → structured role brief → readiness score → prioritized follow-up questions → clarification loop → approved sourcing brief → TAT estimate**

Candidate sourcing, matching, pipeline checks, and outreach are planned as downstream extensions.


# 1. Problem Statement and Success Criteria

## The Problem

Recruiters frequently begin sourcing with incomplete or misaligned hiring information.

A job description may not clearly establish:

* which requirements are non-negotiable;
* where the hiring manager is flexible;
* which skills or experiences are acceptable alternatives;
* whether location, level, sponsorship, compensation, or timing constraints apply;
* what evidence should be tested during candidate screening.

These gaps are often discovered only after profiles have been rejected or candidates have already been screened. This creates avoidable recruiter rework, repeated hiring-manager clarification, inconsistent sourcing, and slower time to shortlist.

## Product Objective

S.O.N.A.R. determines whether a role is ready to source.

It converts fragmented hiring information into a structured role brief, identifies decision-critical gaps, asks prioritized follow-up questions, and produces an approved sourcing specification.

## Primary User Story

> As a recruiter, I want to know whether a role is sufficiently defined to begin sourcing, what information is still missing, and what I should clarify with the hiring manager before investing time in the search.

## Core Outputs

* Sourcing Readiness Score
* Readiness status
* Blocking and high-impact gaps
* Ambiguities and contradictions
* Prioritized hiring-manager questions
* Approved sourcing brief
* Estimated TAT to first qualified shortlist
* Key delay drivers and forecast confidence

## Success Criteria

The prototype should:

1. extract role requirements accurately from a JD and intake notes;
2. distinguish mandatory, preferred, flexible, alternative, and unknown criteria;
3. identify missing information that materially affects sourcing;
4. detect contradictions between the JD and intake discussion;
5. prioritize the next most valuable clarification questions;
6. update the brief and readiness score when new information is added;
7. preserve recruiter control through an explicit approval step;
8. produce a transparent TAT estimate with stated assumptions and uncertainty.

## Evaluation Metrics

### Extraction Quality

* Field-level precision
* Field-level recall
* Field-level F1 score
* Structured-output validity

### Diagnostic Quality

* Blocking-gap detection accuracy
* Contradiction detection accuracy
* False-positive gap rate
* Question relevance and priority accuracy

### Reliability and Safety

* Unsupported-claim rate
* Evidence-grounding rate
* Output consistency across repeated runs
* Protected-attribute exclusion rate
* Human-approval compliance

### Business and Workflow Value

* Time required to create a sourcing-ready brief
* Number of recruiter edits before approval
* Number of clarification rounds required
* Change in readiness score after new information
* TAT forecast error against benchmark cases


In [1]:
# ============================================================
# 2. ENVIRONMENT SETUP
# ============================================================

%pip install -q \
    "google-genai==2.10.0" \
    "pydantic==2.12.5"

import importlib.metadata as metadata

installed_genai = metadata.version("google-genai")
installed_pydantic = metadata.version("pydantic")

assert installed_genai == "2.10.0"
assert installed_pydantic == "2.12.5"

print("✓ S.O.N.A.R. core environment installed successfully")
print(f"✓ google-genai: {installed_genai}")
print(f"✓ pydantic: {installed_pydantic}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.6 MB/s eta 0:00:00
✓ S.O.N.A.R. core environment installed successfully
✓ google-genai: 2.10.0
✓ pydantic: 2.12.5


In [1]:
# ============================================================
# 2.1 IMPORTS AND ENVIRONMENT VERIFICATION
# ============================================================

import json
import importlib.metadata as metadata

from enum import Enum
from typing import List, Optional

from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError


EXPECTED_VERSIONS = {
    "google-genai": "2.10.0",
    "pydantic": "2.12.5",
}

installed_versions = {
    package: metadata.version(package)
    for package in EXPECTED_VERSIONS
}

for package, expected_version in EXPECTED_VERSIONS.items():
    installed_version = installed_versions[package]

    if installed_version != expected_version:
        raise RuntimeError(
            f"{package} version mismatch: "
            f"expected {expected_version}, found {installed_version}"
        )

print("✓ Imports completed successfully")
print("✓ Environment verified")
print(f"✓ google-genai: {installed_versions['google-genai']}")
print(f"✓ pydantic: {installed_versions['pydantic']}")

✓ Imports completed successfully
✓ Environment verified
✓ google-genai: 2.10.0
✓ pydantic: 2.12.5


In [2]:
# ============================================================
# 3. GEMINI API CONNECTION
# ============================================================

SECRET_NAME = "RecruitingAgent"

try:
    api_key = userdata.get(SECRET_NAME)
except Exception as error:
    raise RuntimeError(
        f"Unable to access the Colab secret '{SECRET_NAME}'. "
        "Open the Secrets panel and enable notebook access."
    ) from error

if not api_key or not api_key.strip():
    raise ValueError(
        f"Colab secret '{SECRET_NAME}' is missing or empty. "
        "Add your Gemini API key in the Secrets panel."
    )

client = genai.Client(api_key=api_key)

print("✓ Gemini client created successfully")
print(f"✓ API key loaded securely from Colab secret: {SECRET_NAME}")

✓ Gemini client created successfully
✓ API key loaded securely from Colab secret: RecruitingAgent


In [3]:
# ============================================================
# 3.1 MODEL CONFIGURATION AND ACCESS TEST
# ============================================================

PRIMARY_MODEL = "gemini-2.5-flash-lite"

FALLBACK_MODELS = [
    PRIMARY_MODEL,
    "gemini-2.5-flash",
    "gemini-2.0-flash-lite",
]

try:
    test_response = client.models.generate_content(
        model=PRIMARY_MODEL,
        contents="Reply with exactly one word: READY"
    )

    response_text = (test_response.text or "").strip()

    if response_text != "READY":
        raise RuntimeError(
            f"Unexpected model response: {response_text!r}"
        )

    print(f"✓ Primary model available: {PRIMARY_MODEL}")
    print("✓ Model access test passed")

except Exception as error:
    print(f"⚠ Primary model test failed: {PRIMARY_MODEL}")
    print("✓ The agent will use retry and fallback logic during analysis")
    print(f"Reason: {error}")

✓ Primary model available: gemini-2.5-flash-lite
✓ Model access test passed


# 4. Sourcing Readiness Agent

S.O.N.A.R. determines whether a recruiting role is sufficiently defined to begin sourcing.

It combines:

* the job description;
* recruiter notes or an approved intake-call transcript;
* subsequent hiring-manager clarification answers.

The agent converts these fragmented inputs into a structured role brief, identifies sourcing-critical gaps, prioritizes the next questions, and supports the recruiter in deciding when the role is ready to launch.

```text
Job Description
      +
Intake Notes or Transcript
      +
Clarification Answers
              ↓
       Role Intake Analyzer
              ↓
      Structured Role Brief
              ↓
   Sourcing Readiness Assessment
              ↓
 Gaps + Contradictions + Next Questions
              ↓
       Recruiter Clarification
              ↓
 Updated Brief + Updated Readiness Score
              ↓
         Recruiter Approval
```

The approved sourcing brief becomes the source of truth for downstream sourcing, screening, matching, and hiring-manager reporting.

In [4]:
# ============================================================
# 4.1 CONTROLLED WORKFLOW VALUES
# ============================================================

class QuestionPriority(str, Enum):
    BLOCKING = "blocking"
    HIGH_IMPACT = "high_impact"
    HELPFUL = "helpful"
    ADMINISTRATIVE = "administrative"


class ReadinessStatus(str, Enum):
    NOT_READY = "not_ready"
    NEEDS_CLARIFICATION = "needs_clarification"
    READY_WITH_ASSUMPTIONS = "ready_with_assumptions"
    READY_TO_SOURCE = "ready_to_source"


class ApprovalStatus(str, Enum):
    DRAFT = "draft"
    PENDING_RECRUITER_REVIEW = "pending_recruiter_review"
    APPROVED = "approved"
    REJECTED_FOR_REVISION = "rejected_for_revision"

print("✓ Controlled workflow values created")

✓ Controlled workflow values created


In [5]:
# ============================================================
# 4.2 SOURCING BRIEF DATA MODEL
# ============================================================

class ClarificationQuestion(BaseModel):
    """
    One recruiter-facing question generated from an unresolved issue.
    """

    question: str
    priority: QuestionPriority
    reason: str
    affected_area: str
    related_fields: List[str] = Field(default_factory=list)


class SourceEvidence(BaseModel):
    """
    Evidence supporting one extracted role requirement.
    """

    evidence_id: str
    field_name: str
    value: str
    source_type: str
    supporting_text: str


class SourcingBrief(BaseModel):
    """
    Versioned structured brief created from the job description,
    intake notes or transcript, and clarification answers.
    """

    # Role definition
    role_title: Optional[str] = None
    team_name: Optional[str] = None
    seniority_levels: List[str] = Field(default_factory=list)
    level_flexibility: Optional[str] = None
    key_responsibilities: List[str] = Field(default_factory=list)

    # Search scope
    locations: List[str] = Field(default_factory=list)
    location_flexibility: Optional[str] = None
    work_arrangement: Optional[str] = None
    employment_type: Optional[str] = None
    number_of_openings: Optional[int] = None
    target_hire_date: Optional[str] = None

    # Eligibility and commercial constraints
    visa_sponsorship: Optional[bool] = None
    relocation_available: Optional[bool] = None
    compensation_range: Optional[str] = None

    # Candidate requirements
    must_have_skills: List[str] = Field(default_factory=list)
    nice_to_have_skills: List[str] = Field(default_factory=list)
    must_have_qualifications: List[str] = Field(default_factory=list)
    nice_to_have_qualifications: List[str] = Field(default_factory=list)

    # Hiring-manager interpretation
    non_negotiable_requirements: List[str] = Field(default_factory=list)
    flexible_requirements: List[str] = Field(default_factory=list)
    acceptable_alternatives: List[str] = Field(default_factory=list)
    disqualifying_conditions: List[str] = Field(default_factory=list)
    sourcing_caveats: List[str] = Field(default_factory=list)

    # Sourcing direction
    target_companies: List[str] = Field(default_factory=list)
    target_job_titles: List[str] = Field(default_factory=list)
    adjacent_profile_types: List[str] = Field(default_factory=list)

    # Candidate assessment
    candidate_screening_questions: List[str] = Field(default_factory=list)
    expected_screening_evidence: List[str] = Field(default_factory=list)

    # Diagnostic findings
    missing_information: List[str] = Field(default_factory=list)
    ambiguities: List[str] = Field(default_factory=list)
    contradictions: List[str] = Field(default_factory=list)
    clarification_questions: List[ClarificationQuestion] = Field(
        default_factory=list
    )
    source_evidence: List[SourceEvidence] = Field(default_factory=list)

    # Workflow state
    brief_version: int = 1
    approval_status: ApprovalStatus = ApprovalStatus.DRAFT


schema_fields = set(SourcingBrief.model_fields)

critical_fields = {
    "role_title",
    "must_have_skills",
    "clarification_questions",
    "source_evidence",
    "brief_version",
    "approval_status",
}

missing_fields = critical_fields - schema_fields

if missing_fields:
    raise RuntimeError(
        f"SourcingBrief schema missing critical fields: "
        f"{sorted(missing_fields)}"
    )

print("✓ SourcingBrief schema created and verified")
print(f"✓ Total schema fields: {len(schema_fields)}")

✓ SourcingBrief schema created and verified
✓ Total schema fields: 35


In [6]:
# ============================================================
# 4.3 CONTROLLED DEMONSTRATION CASE
# ============================================================

# This controlled case is used to demonstrate and evaluate the workflow.
# It is not a hard-coded product limitation.
# The final Streamlit application will accept any recruiter-provided
# job description, intake notes, transcript, or clarification answers.

DEMO_CASE = {
    "case_id": "senior_data_engineer_001",
    "case_name": "Senior Data Engineer Readiness Review",

    "job_description": """
Senior Data Engineer

We are looking for a Senior Data Engineer to design and operate
large-scale data pipelines supporting analytics and machine learning.

Responsibilities:
- Build reliable batch and real-time data pipelines
- Work with engineering, analytics, and machine learning teams
- Improve data quality, reliability, and platform scalability

Minimum qualifications:
- 5+ years of data engineering or backend engineering experience
- Strong Python and SQL skills
- Experience with distributed data-processing systems
- Experience with cloud infrastructure

Preferred qualifications:
- Experience with Spark, Kafka, or Flink
- Experience with Airflow or dbt
- Experience in fintech or another high-scale technology environment
""",

    "intake_context": """
The role sits within the Data Platform team.

The hiring manager prefers a senior candidate but may consider a
strong mid-level candidate.

Kafka experience is mandatory because the team owns real-time
data pipelines. Spark or Flink experience is useful, and either
one is acceptable.

Python and SQL are non-negotiable.

The role can be based in San Francisco or Seattle and follows a
hybrid working arrangement.

Stripe, Uber, Airbnb, and Block are useful source companies, but
the manager is open to candidates from other high-scale technology
companies.

This is a replacement hire with one opening.

Visa sponsorship, compensation, and the target hiring date have
not yet been confirmed.

During screening, candidates should explain a production data
pipeline they designed, including its scale, reliability, and
technical trade-offs.
""",

    "clarification_answers": "",

    # Used later for evaluation only.
    # These values are not sent to Gemini.
    "expected_signals": {
        "must_have_skills": [
            "Python",
            "SQL",
            "Kafka",
        ],
        "acceptable_alternatives": [
            "Spark or Flink",
        ],
        "locations": [
            "San Francisco",
            "Seattle",
        ],
        "unresolved_topics": [
            "visa sponsorship",
            "compensation",
            "target hiring date",
        ],
    },
}

required_demo_keys = {
    "case_id",
    "case_name",
    "job_description",
    "intake_context",
    "clarification_answers",
    "expected_signals",
}

missing_demo_keys = required_demo_keys - set(DEMO_CASE)

if missing_demo_keys:
    raise RuntimeError(
        f"DEMO_CASE is missing required keys: "
        f"{sorted(missing_demo_keys)}"
    )

print("✓ Controlled demonstration case loaded")
print(f"✓ Case: {DEMO_CASE['case_name']}")
print(f"✓ Case ID: {DEMO_CASE['case_id']}")

✓ Controlled demonstration case loaded
✓ Case: Senior Data Engineer Readiness Review
✓ Case ID: senior_data_engineer_001


In [7]:
# ============================================================
# 4.4 ROLE INTAKE ANALYZER INSTRUCTIONS
# ============================================================

SOURCING_READINESS_INSTRUCTIONS = """
You are S.O.N.A.R.'s Role Intake Analyzer.

Your job is to convert unstructured recruiting information into a
structured, evidence-backed SourcingBrief.

The input may contain:

1. a job description;
2. recruiter notes or an intake-call transcript;
3. later clarification answers from the recruiter or hiring manager.

You are responsible for extraction, normalization, gap detection,
contradiction detection, and clarification planning.

You are not responsible for calculating the numerical readiness score,
deciding whether sourcing may begin, or approving the role.

SOURCE DISCIPLINE

1. Use only information supported by the provided inputs.
2. Do not invent requirements, policies, locations, compensation,
   timelines, hiring preferences, target companies, or constraints.
3. Treat information that is not provided as unknown.
4. Do not convert an assumption into a confirmed fact.
5. Preserve the meaning of the source text without overstating it.
6. For every important extracted fact, create a source_evidence item.
7. Each source_evidence item must contain:
   - a unique evidence_id;
   - the destination field_name;
   - the extracted value;
   - the source_type;
   - a short supporting excerpt from the input.
8. Use source_type values such as:
   - job_description;
   - intake_context;
   - clarification_answer.

REQUIREMENT CLASSIFICATION

9. Distinguish clearly between:
   - must-have requirements;
   - preferred requirements;
   - non-negotiable requirements;
   - flexible requirements;
   - acceptable alternatives;
   - disqualifying conditions.
10. A flexible requirement means the hiring manager may relax it.
11. An acceptable alternative means one skill, title, qualification,
    technology, industry, or experience may substitute for another.
12. Classify programming languages, tools, technologies, frameworks,
    platforms, and infrastructure capabilities under skills.
13. Classify years of experience, degrees, certifications, and broader
    experience requirements under qualifications.
14. Treat requirements listed under minimum qualifications as confirmed
    baseline requirements.
15. Treat requirements listed under preferred qualifications as
    nice-to-have skills or nice-to-have qualifications.
16. Infer nominal seniority from an explicit title such as Senior,
    Staff, Lead, Principal, Junior, Manager, or Director.
17. Do not infer level flexibility from the title alone.
18. Do not place screening expectations under sourcing-direction fields.
19. Convert screening expectations into:
    - candidate_screening_questions; or
    - expected_screening_evidence.

SOURCE RECONCILIATION

20. When later inputs add detail without contradicting earlier inputs,
    preserve the more specific information.
21. When clarification answers explicitly revise an earlier requirement,
    use the latest explicit answer and preserve evidence for it.
22. When two sources conflict materially, do not silently choose one.
23. Record each material conflict under contradictions.
24. Generate a clarification question for each unresolved material
    contradiction.
25. A contradiction is material when it could change:
    - candidate eligibility;
    - the available candidate pool;
    - search geography;
    - sourcing strategy;
    - screening criteria;
    - expected sourcing effort;
    - expected timeline.

GAP AND AMBIGUITY DETECTION

26. Identify missing information only when it could materially affect:
    - candidate eligibility;
    - search scope;
    - recruiter effort;
    - screening consistency;
    - hiring timeline.
27. Relevant topics may include:
    - location;
    - work arrangement;
    - seniority flexibility;
    - mandatory skills;
    - acceptable alternatives;
    - number of openings;
    - visa sponsorship;
    - compensation;
    - target hiring date;
    - screening evidence.
28. Do not list every empty schema field as missing.
29. Do not treat optional sourcing preferences as mandatory gaps.
30. Record unclear or open-to-interpretation language under ambiguities.

CLARIFICATION PLANNING

31. Generate no more than five clarification questions per analysis.
32. Prioritize questions that materially affect candidate eligibility,
    search scope, or sourcing effort.
33. Classify each question as:

    blocking:
    The role should not proceed until the issue is resolved.

    high_impact:
    The answer could substantially change the candidate pool,
    search strategy, or screening process.

    helpful:
    The answer would improve precision or efficiency but does not
    prevent initial sourcing.

    administrative:
    Useful for process management but not central to sourcing readiness.

34. Each clarification question must contain:
    - a concise recruiter-friendly question;
    - its priority;
    - why the answer matters;
    - the affected area;
    - the related SourcingBrief fields.
35. Avoid vague questions such as "Can you clarify the role?"
36. Avoid asking for information already present in the inputs.
37. Avoid low-value administrative questions when more material gaps
    remain unresolved.

OUTPUT BOUNDARY

38. Return only information that fits the SourcingBrief schema.
39. Do not calculate a numerical Sourcing Readiness Score.
40. Do not decide whether sourcing can begin.
41. Do not set or infer recruiter approval.
42. Keep approval_status as draft unless explicit recruiter approval is
    provided in the input.
43. Set brief_version to 1 during initial extraction. Later Python
    workflow logic will control version increments.
44. Do not use protected personal characteristics or proxies for them
    as sourcing or screening criteria.
"""

if not SOURCING_READINESS_INSTRUCTIONS.strip():
    raise RuntimeError("Role Intake Analyzer instructions are empty.")

print("✓ Role Intake Analyzer instructions created and verified")

✓ Role Intake Analyzer instructions created and verified


In [8]:
# ============================================================
# 4.5 RELIABLE SOURCING BRIEF ANALYSIS
# ============================================================

import random
import time


TEMPORARY_ERROR_MARKERS = (
    "429",
    "503",
    "unavailable",
    "resource_exhausted",
    "high demand",
    "overloaded",
    "temporarily unavailable",
    "deadline exceeded",
)


def is_temporary_model_error(error: Exception) -> bool:
    """
    Identify API errors that may succeed after retrying.
    """

    error_text = str(error).lower()

    return any(
        marker.lower() in error_text
        for marker in TEMPORARY_ERROR_MARKERS
    )


def analyze_sourcing_brief(
    job_description: str,
    intake_context: str = "",
    clarification_answers: str = "",
    max_retries_per_model: int = 3,
) -> tuple[SourcingBrief, str]:
    """
    Convert recruiter-provided information into a validated SourcingBrief.

    The function:
    1. validates the inputs;
    2. sends the recruiting context to Gemini;
    3. validates the structured response;
    4. retries temporary API failures;
    5. falls back to another configured model when required.

    Returns:
        A tuple containing:
        - the validated SourcingBrief;
        - the Gemini model that successfully completed the analysis.
    """

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    if not isinstance(job_description, str):
        raise TypeError("job_description must be a string.")

    if not job_description.strip():
        raise ValueError("A job description is required.")

    if not isinstance(intake_context, str):
        raise TypeError("intake_context must be a string.")

    if not isinstance(clarification_answers, str):
        raise TypeError("clarification_answers must be a string.")

    if max_retries_per_model < 1:
        raise ValueError(
            "max_retries_per_model must be at least 1."
        )

    if not FALLBACK_MODELS:
        raise RuntimeError(
            "No Gemini models have been configured."
        )

    # --------------------------------------------------------
    # BUILD THE MODEL INPUT
    # --------------------------------------------------------

    combined_input = f"""
JOB DESCRIPTION
---------------
{job_description.strip()}

RECRUITER NOTES OR INTAKE TRANSCRIPT
------------------------------------
{intake_context.strip() or "No intake context provided."}

CLARIFICATION ANSWERS
---------------------
{clarification_answers.strip() or "No clarification answers provided."}
"""

    last_error = None
    attempted_models = []

    # --------------------------------------------------------
    # MODEL RETRY AND FALLBACK LOOP
    # --------------------------------------------------------

    for model_name in FALLBACK_MODELS:
        attempted_models.append(model_name)

        for attempt in range(1, max_retries_per_model + 1):

            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=combined_input,
                    config={
                        "system_instruction":
                            SOURCING_READINESS_INSTRUCTIONS,
                        "response_mime_type": "application/json",
                        "response_schema": SourcingBrief,
                        "temperature": 0.1,
                    },
                )

                response_text = (response.text or "").strip()

                if not response_text:
                    raise RuntimeError(
                        "The model returned an empty response."
                    )

                brief = SourcingBrief.model_validate_json(
                    response_text
                )

                return brief, model_name

            except ValidationError as error:
                raise RuntimeError(
                    "Gemini returned a response that did not match "
                    "the SourcingBrief schema."
                ) from error

            except Exception as error:
                last_error = error

                if not is_temporary_model_error(error):
                    raise RuntimeError(
                        f"Sourcing brief analysis failed with "
                        f"{model_name}: {error}"
                    ) from error

                if attempt < max_retries_per_model:
                    wait_seconds = (
                        2 ** attempt
                        + random.uniform(0.0, 1.0)
                    )

                    print(
                        f"Temporary issue with {model_name}. "
                        f"Retrying in {wait_seconds:.1f} seconds..."
                    )

                    time.sleep(wait_seconds)

                else:
                    print(
                        f"{model_name} remained unavailable. "
                        "Trying the next configured model..."
                    )

    # --------------------------------------------------------
    # FINAL FAILURE
    # --------------------------------------------------------

    attempted_models_text = ", ".join(attempted_models)

    raise RuntimeError(
        "S.O.N.A.R. could not reach an available Gemini model after "
        f"trying: {attempted_models_text}. "
        "This is usually a temporary service or quota issue. "
        f"Last error: {last_error}"
    )


# Verify that the analyzer was created successfully.
if not callable(analyze_sourcing_brief):
    raise RuntimeError(
        "analyze_sourcing_brief was not created successfully."
    )

print("✓ Reliable sourcing brief analyzer created and verified")

✓ Reliable sourcing brief analyzer created and verified


In [9]:
# ============================================================
# 4.6 CREATE BRIEF V1 FROM THE JOB DESCRIPTION
# ============================================================

print("Analyzing the job description...")

try:
    brief_v1, brief_v1_model = analyze_sourcing_brief(
        job_description=DEMO_CASE["job_description"]
    )

except Exception as error:
    raise RuntimeError(
        "Brief v1 could not be created from the demonstration JD."
    ) from error


# ------------------------------------------------------------
# VERIFY THE RESULT
# ------------------------------------------------------------

if not isinstance(brief_v1, SourcingBrief):
    raise TypeError(
        "The analyzer did not return a valid SourcingBrief."
    )

if brief_v1.brief_version != 1:
    raise RuntimeError(
        f"Expected brief_version 1, found {brief_v1.brief_version}."
    )

if not brief_v1.role_title:
    raise RuntimeError(
        "The role title was not extracted from the job description."
    )


# ------------------------------------------------------------
# SUMMARY OUTPUT
# ------------------------------------------------------------

print("\n✓ Brief v1 created successfully")
print(f"✓ Model used: {brief_v1_model}")
print(f"✓ Role identified: {brief_v1.role_title}")
print(f"✓ Evidence items captured: {len(brief_v1.source_evidence)}")
print(
    "✓ Clarification questions generated: "
    f"{len(brief_v1.clarification_questions)}"
)
print(f"✓ Approval status: {brief_v1.approval_status.value}")

Analyzing the job description...
Temporary issue with gemini-2.5-flash-lite. Retrying in 2.8 seconds...

✓ Brief v1 created successfully
✓ Model used: gemini-2.5-flash-lite
✓ Role identified: Senior Data Engineer
✓ Evidence items captured: 16
✓ Clarification questions generated: 0
✓ Approval status: draft


In [10]:
# ============================================================
# 4.7 RECRUITER-FRIENDLY BRIEF REVIEW
# ============================================================

def print_list_section(
    title: str,
    items: List[str],
    empty_message: str
) -> None:
    """
    Print a clean recruiter-facing section.
    """

    print(f"\n{title}")
    print("-" * len(title))

    if items:
        for item in items:
            print(f"• {item}")
    else:
        print(empty_message)


print("=" * 68)
print("S.O.N.A.R. ROLE REVIEW")
print("=" * 68)

print(f"\nRole: {brief_v1.role_title or 'Not confirmed'}")

if brief_v1.seniority_levels:
    print(
        "Seniority: "
        + ", ".join(brief_v1.seniority_levels)
    )
else:
    print("Seniority: Not confirmed")

print_list_section(
    title="CORE RESPONSIBILITIES",
    items=brief_v1.key_responsibilities,
    empty_message="No core responsibilities were identified.",
)

print_list_section(
    title="CONFIRMED MUST-HAVE SKILLS",
    items=brief_v1.must_have_skills,
    empty_message="No must-have skills were confirmed.",
)

print_list_section(
    title="PREFERRED SKILLS",
    items=brief_v1.nice_to_have_skills,
    empty_message="No preferred skills were identified.",
)

print_list_section(
    title="MUST-HAVE QUALIFICATIONS",
    items=brief_v1.must_have_qualifications,
    empty_message="No must-have qualifications were confirmed.",
)

print_list_section(
    title="PREFERRED QUALIFICATIONS",
    items=brief_v1.nice_to_have_qualifications,
    empty_message="No preferred qualifications were identified.",
)

print("\nTOP CLARIFICATION QUESTIONS")
print("---------------------------")

if brief_v1.clarification_questions:
    for index, item in enumerate(
        brief_v1.clarification_questions,
        start=1,
    ):
        print(
            f"\n{index}. [{item.priority.value.upper()}] "
            f"{item.question}"
        )
        print(f"   Why it matters: {item.reason}")
        print(f"   Affected area: {item.affected_area}")

        if item.related_fields:
            print(
                "   Related fields: "
                + ", ".join(item.related_fields)
            )
else:
    print("No material clarification questions were generated.")

print("\nTRACEABILITY")
print("------------")
print(
    f"{len(brief_v1.source_evidence)} evidence item(s) captured "
    "from the source text."
)

print("\nWORKFLOW STATUS")
print("---------------")
print(f"Brief version: {brief_v1.brief_version}")
print(f"Approval status: {brief_v1.approval_status.value}")
print(
    "Readiness will be calculated next by the deterministic "
    "Sourcing Readiness Tool."
)

S.O.N.A.R. ROLE REVIEW

Role: Senior Data Engineer
Seniority: Senior

CORE RESPONSIBILITIES
---------------------
• Build reliable batch and real-time data pipelines
• Work with engineering, analytics, and machine learning teams
• Improve data quality, reliability, and platform scalability

CONFIRMED MUST-HAVE SKILLS
--------------------------
No must-have skills were confirmed.

PREFERRED SKILLS
----------------
No preferred skills were identified.

MUST-HAVE QUALIFICATIONS
------------------------
• 5+ years of data engineering or backend engineering experience

PREFERRED QUALIFICATIONS
------------------------
No preferred qualifications were identified.

TOP CLARIFICATION QUESTIONS
---------------------------
No material clarification questions were generated.

TRACEABILITY
------------
16 evidence item(s) captured from the source text.

WORKFLOW STATUS
---------------
Brief version: 1
Approval status: draft
Readiness will be calculated next by the deterministic Sourcing Readiness 

# 5. Sourcing Readiness Tool

The Role Intake Analyzer converts unstructured recruiting information into a structured and evidence-backed `SourcingBrief`.

The Sourcing Readiness Tool now evaluates whether that brief contains enough information to begin sourcing consistently, transparently, and with appropriate recruiter control.

The readiness score is calculated with deterministic Python rules rather than generated by the language model. This separation ensures that:

- the same structured brief always produces the same score;
- blocking gaps are treated consistently;
- recruiters can understand why points were gained or lost;
- scoring logic can be reviewed and adjusted;
- the language model cannot arbitrarily change the result.

The tool evaluates five dimensions:

1. **Role Definition**
2. **Candidate Eligibility**
3. **Search Parameters**
4. **Assessment Alignment**
5. **Hiring Logistics**

Each dimension contributes to a total score of 100.

A separate blocking-gap check prevents a role from being marked ready when a sourcing-critical decision remains unresolved.

```text
Unstructured recruiting inputs
              ↓
     Role Intake Analyzer
              ↓
       Structured Brief
              ↓
   Sourcing Readiness Tool
              ↓
Score + Blocking Gaps + Decision

In [11]:
# ============================================================
# 5.1 READINESS ASSESSMENT DATA MODEL
# ============================================================

class ReadinessDimension(BaseModel):
    """
    Score and explanation for one sourcing-readiness dimension.
    """

    dimension_name: str
    score: int = Field(ge=0)
    max_score: int = Field(gt=0)
    completed_checks: List[str] = Field(default_factory=list)
    unresolved_checks: List[str] = Field(default_factory=list)
    rationale: str


class ReadinessAssessment(BaseModel):
    """
    Deterministic diagnostic result created from a SourcingBrief.
    """

    total_score: int = Field(ge=0, le=100)
    readiness_status: ReadinessStatus

    dimensions: List[ReadinessDimension] = Field(
        default_factory=list
    )

    blocking_gaps: List[str] = Field(default_factory=list)
    high_impact_gaps: List[str] = Field(default_factory=list)
    helpful_gaps: List[str] = Field(default_factory=list)

    strengths: List[str] = Field(default_factory=list)
    recruiter_message: str

    can_begin_sourcing: bool = False
    requires_recruiter_approval: bool = True


# Verify that the assessment schema contains the fields
# required by the scoring engine and recruiter interface.
assessment_fields = set(ReadinessAssessment.model_fields)

required_assessment_fields = {
    "total_score",
    "readiness_status",
    "dimensions",
    "blocking_gaps",
    "high_impact_gaps",
    "recruiter_message",
    "can_begin_sourcing",
    "requires_recruiter_approval",
}

missing_assessment_fields = (
    required_assessment_fields - assessment_fields
)

if missing_assessment_fields:
    raise RuntimeError(
        "ReadinessAssessment is missing required fields: "
        f"{sorted(missing_assessment_fields)}"
    )

print("✓ Readiness assessment schema created and verified")
print(
    f"✓ Total assessment fields: "
    f"{len(assessment_fields)}"
)

✓ Readiness assessment schema created and verified
✓ Total assessment fields: 10


In [12]:
# ============================================================
# 5.2 READINESS SCORING CONFIGURATION
# ============================================================

READINESS_WEIGHTS = {
    "role_definition": 20,
    "candidate_eligibility": 25,
    "search_parameters": 20,
    "assessment_alignment": 20,
    "hiring_logistics": 15,
}

READINESS_THRESHOLDS = {
    "ready_to_source": 85,
    "ready_with_assumptions": 70,
    "needs_clarification": 50,
}

EXPECTED_DIMENSIONS = {
    "role_definition",
    "candidate_eligibility",
    "search_parameters",
    "assessment_alignment",
    "hiring_logistics",
}


# ------------------------------------------------------------
# CONFIGURATION VALIDATION
# ------------------------------------------------------------

configured_dimensions = set(READINESS_WEIGHTS)

if configured_dimensions != EXPECTED_DIMENSIONS:
    missing_dimensions = EXPECTED_DIMENSIONS - configured_dimensions
    unexpected_dimensions = configured_dimensions - EXPECTED_DIMENSIONS

    raise RuntimeError(
        "Readiness dimensions are misconfigured. "
        f"Missing: {sorted(missing_dimensions)}. "
        f"Unexpected: {sorted(unexpected_dimensions)}."
    )

total_weight = sum(READINESS_WEIGHTS.values())

if total_weight != 100:
    raise RuntimeError(
        f"Readiness weights must total 100, found {total_weight}."
    )

if not (
    0
    < READINESS_THRESHOLDS["needs_clarification"]
    < READINESS_THRESHOLDS["ready_with_assumptions"]
    < READINESS_THRESHOLDS["ready_to_source"]
    <= 100
):
    raise RuntimeError(
        "Readiness thresholds are not in a valid ascending order."
    )


print("✓ Readiness scoring configuration created and verified")
print(f"✓ Total available score: {total_weight}")
print(
    "✓ Status thresholds: "
    f"{READINESS_THRESHOLDS['needs_clarification']} / "
    f"{READINESS_THRESHOLDS['ready_with_assumptions']} / "
    f"{READINESS_THRESHOLDS['ready_to_source']}"
)

✓ Readiness scoring configuration created and verified
✓ Total available score: 100
✓ Status thresholds: 50 / 70 / 85


In [13]:
# ============================================================
# 5.3 GAP CLASSIFICATION RULES
# ============================================================

# These rules classify missing information by its likely impact on
# sourcing. They do not calculate the score directly. The scoring
# function will evaluate them against each SourcingBrief.

BLOCKING_GAP_RULES = {
    "role_title": (
        "The role title or role family has not been confirmed."
    ),
    "seniority_levels": (
        "The required seniority level has not been confirmed."
    ),
    "key_responsibilities": (
        "The core responsibilities are not sufficiently defined."
    ),
    "must_have_skills": (
        "The mandatory skills are not sufficiently defined."
    ),
    "locations": (
        "The permitted hiring location has not been confirmed."
    ),
    "work_arrangement": (
        "The remote, hybrid, or on-site expectation has not been confirmed."
    ),
}


HIGH_IMPACT_GAP_RULES = {
    "level_flexibility": (
        "It is unclear whether candidates at adjacent seniority levels "
        "may be considered."
    ),
    "non_negotiable_requirements": (
        "The hiring manager's non-negotiable requirements have not been "
        "explicitly documented."
    ),
    "acceptable_alternatives": (
        "Acceptable substitutes for skills, technologies, or experience "
        "have not been clarified."
    ),
    "visa_sponsorship": (
        "Visa sponsorship availability has not been confirmed."
    ),
    "compensation_range": (
        "The approved compensation range has not been provided."
    ),
    "number_of_openings": (
        "The number of openings has not been confirmed."
    ),
    "target_hire_date": (
        "The target hiring date has not been confirmed."
    ),
}


HELPFUL_GAP_RULES = {
    "team_name": (
        "The hiring team or organizational context has not been provided."
    ),
    "target_job_titles": (
        "Target and adjacent job titles have not yet been defined."
    ),
    "target_companies": (
        "Example source companies or company profiles have not been defined."
    ),
    "candidate_screening_questions": (
        "Candidate screening questions have not yet been documented."
    ),
    "expected_screening_evidence": (
        "The evidence candidates should provide during screening has not "
        "been documented."
    ),
}


# ------------------------------------------------------------
# CONFIGURATION VALIDATION
# ------------------------------------------------------------

all_rule_fields = (
    set(BLOCKING_GAP_RULES)
    | set(HIGH_IMPACT_GAP_RULES)
    | set(HELPFUL_GAP_RULES)
)

unknown_rule_fields = all_rule_fields - set(SourcingBrief.model_fields)

if unknown_rule_fields:
    raise RuntimeError(
        "Gap rules reference fields that do not exist in SourcingBrief: "
        f"{sorted(unknown_rule_fields)}"
    )

overlapping_fields = (
    set(BLOCKING_GAP_RULES) & set(HIGH_IMPACT_GAP_RULES)
) | (
    set(BLOCKING_GAP_RULES) & set(HELPFUL_GAP_RULES)
) | (
    set(HIGH_IMPACT_GAP_RULES) & set(HELPFUL_GAP_RULES)
)

if overlapping_fields:
    raise RuntimeError(
        "A field cannot appear in more than one gap category: "
        f"{sorted(overlapping_fields)}"
    )


print("✓ Gap classification rules created and verified")
print(f"✓ Blocking rules: {len(BLOCKING_GAP_RULES)}")
print(f"✓ High-impact rules: {len(HIGH_IMPACT_GAP_RULES)}")
print(f"✓ Helpful rules: {len(HELPFUL_GAP_RULES)}")

✓ Gap classification rules created and verified
✓ Blocking rules: 6
✓ High-impact rules: 7
✓ Helpful rules: 5


In [14]:
# ============================================================
# 5.4 MISSING-VALUE HELPER
# ============================================================

def is_missing(value) -> bool:
    """
    Return True when a SourcingBrief value is meaningfully empty.

    Confirmed boolean values such as False are treated as known,
    not missing.
    """

    if value is None:
        return True

    if isinstance(value, str):
        return not value.strip()

    if isinstance(value, (list, tuple, set, dict)):
        return len(value) == 0

    return False


# ------------------------------------------------------------
# HELPER VERIFICATION
# ------------------------------------------------------------

missing_value_tests = {
    "none_is_missing": is_missing(None) is True,
    "empty_string_is_missing": is_missing("") is True,
    "whitespace_is_missing": is_missing("   ") is True,
    "empty_list_is_missing": is_missing([]) is True,
    "populated_list_is_known": is_missing(["Python"]) is False,
    "false_boolean_is_known": is_missing(False) is False,
    "zero_is_known": is_missing(0) is False,
}

failed_tests = [
    test_name
    for test_name, passed in missing_value_tests.items()
    if not passed
]

if failed_tests:
    raise RuntimeError(
        "Missing-value helper failed verification: "
        f"{failed_tests}"
    )

print("✓ Missing-value helper created and verified")
print(f"✓ Verification checks passed: {len(missing_value_tests)}")

✓ Missing-value helper created and verified
✓ Verification checks passed: 7


In [15]:
# ============================================================
# 5.5 DETERMINISTIC SOURCING READINESS SCORING
# ============================================================

def calculate_sourcing_readiness(
    brief: SourcingBrief
) -> ReadinessAssessment:
    """
    Calculate an explainable and deterministic readiness assessment
    from a validated SourcingBrief.

    Gemini extracts and structures the recruiting information.
    This Python tool applies fixed scoring and blocking rules.
    """

    if not isinstance(brief, SourcingBrief):
        raise TypeError("brief must be a validated SourcingBrief.")

    # --------------------------------------------------------
    # HELPER FUNCTIONS
    # --------------------------------------------------------

    def completed(value) -> bool:
        return not is_missing(value)

    def unique(items: List[str]) -> List[str]:
        """
        Remove duplicates while preserving order.
        """
        return list(dict.fromkeys(items))

    def build_dimension(
        dimension_name: str,
        checks: dict
    ) -> ReadinessDimension:
        """
        Build one scored readiness dimension.

        checks format:
        {
            "Check description": (is_complete, point_value)
        }
        """

        completed_checks = [
            check_name
            for check_name, (is_complete, _) in checks.items()
            if is_complete
        ]

        unresolved_checks = [
            check_name
            for check_name, (is_complete, _) in checks.items()
            if not is_complete
        ]

        score = sum(
            points
            for is_complete, points in checks.values()
            if is_complete
        )

        max_score = sum(
            points
            for _, points in checks.values()
        )

        return ReadinessDimension(
            dimension_name=dimension_name,
            score=score,
            max_score=max_score,
            completed_checks=completed_checks,
            unresolved_checks=unresolved_checks,
            rationale=(
                f"{len(completed_checks)} of "
                f"{len(checks)} readiness checks completed."
            ),
        )

    # --------------------------------------------------------
    # 1. CLASSIFY UNRESOLVED GAPS
    # --------------------------------------------------------

    blocking_gaps = [
        message
        for field_name, message in BLOCKING_GAP_RULES.items()
        if is_missing(getattr(brief, field_name))
    ]

    high_impact_gaps = [
        message
        for field_name, message in HIGH_IMPACT_GAP_RULES.items()
        if is_missing(getattr(brief, field_name))
    ]

    helpful_gaps = [
        message
        for field_name, message in HELPFUL_GAP_RULES.items()
        if is_missing(getattr(brief, field_name))
    ]

    blocking_gaps.extend(
        f"Unresolved contradiction: {contradiction}"
        for contradiction in brief.contradictions
    )

    high_impact_gaps.extend(
        f"Unresolved ambiguity: {ambiguity}"
        for ambiguity in brief.ambiguities
    )

    blocking_gaps = unique(blocking_gaps)
    high_impact_gaps = unique(high_impact_gaps)
    helpful_gaps = unique(helpful_gaps)

    # --------------------------------------------------------
    # 2. ROLE DEFINITION: 20 POINTS
    # --------------------------------------------------------

    role_definition = build_dimension(
        "Role Definition",
        {
            "Role title confirmed": (
                completed(brief.role_title),
                5,
            ),
            "Seniority confirmed": (
                completed(brief.seniority_levels),
                5,
            ),
            "Core responsibilities defined": (
                completed(brief.key_responsibilities),
                7,
            ),
            "Team context provided": (
                completed(brief.team_name),
                3,
            ),
        },
    )

    # --------------------------------------------------------
    # 3. CANDIDATE ELIGIBILITY: 25 POINTS
    # --------------------------------------------------------

    candidate_eligibility = build_dimension(
        "Candidate Eligibility",
        {
            "Must-have skills defined": (
                completed(brief.must_have_skills),
                7,
            ),
            "Must-have qualifications defined": (
                completed(brief.must_have_qualifications),
                5,
            ),
            "Non-negotiables documented": (
                completed(brief.non_negotiable_requirements),
                5,
            ),
            "Level flexibility clarified": (
                completed(brief.level_flexibility),
                4,
            ),
            "Acceptable alternatives documented": (
                completed(brief.acceptable_alternatives),
                4,
            ),
        },
    )

    # --------------------------------------------------------
    # 4. SEARCH PARAMETERS: 20 POINTS
    # --------------------------------------------------------

    sourcing_direction_available = any(
        [
            completed(brief.target_companies),
            completed(brief.target_job_titles),
            completed(brief.adjacent_profile_types),
        ]
    )

    search_parameters = build_dimension(
        "Search Parameters",
        {
            "Hiring locations confirmed": (
                completed(brief.locations),
                6,
            ),
            "Work arrangement confirmed": (
                completed(brief.work_arrangement),
                5,
            ),
            "Employment type confirmed": (
                completed(brief.employment_type),
                3,
            ),
            "Target titles defined": (
                completed(brief.target_job_titles),
                3,
            ),
            "Sourcing direction provided": (
                sourcing_direction_available,
                3,
            ),
        },
    )

    # --------------------------------------------------------
    # 5. ASSESSMENT ALIGNMENT: 20 POINTS
    # --------------------------------------------------------

    assessment_alignment = build_dimension(
        "Assessment Alignment",
        {
            "Screening questions documented": (
                completed(brief.candidate_screening_questions),
                7,
            ),
            "Expected screening evidence documented": (
                completed(brief.expected_screening_evidence),
                7,
            ),
            "Core responsibilities support assessment": (
                completed(brief.key_responsibilities),
                6,
            ),
        },
    )

    # --------------------------------------------------------
    # 6. HIRING LOGISTICS: 15 POINTS
    # --------------------------------------------------------

    hiring_logistics = build_dimension(
        "Hiring Logistics",
        {
            "Number of openings confirmed": (
                completed(brief.number_of_openings),
                3,
            ),
            "Target hiring date confirmed": (
                completed(brief.target_hire_date),
                3,
            ),
            "Visa sponsorship confirmed": (
                completed(brief.visa_sponsorship),
                3,
            ),
            "Compensation range confirmed": (
                completed(brief.compensation_range),
                3,
            ),
            "Location flexibility clarified": (
                completed(brief.location_flexibility),
                3,
            ),
        },
    )

    dimensions = [
        role_definition,
        candidate_eligibility,
        search_parameters,
        assessment_alignment,
        hiring_logistics,
    ]

    # --------------------------------------------------------
    # 7. VERIFY DIMENSION MAX SCORES
    # --------------------------------------------------------

    expected_max_scores = {
        "Role Definition": READINESS_WEIGHTS["role_definition"],
        "Candidate Eligibility": READINESS_WEIGHTS[
            "candidate_eligibility"
        ],
        "Search Parameters": READINESS_WEIGHTS[
            "search_parameters"
        ],
        "Assessment Alignment": READINESS_WEIGHTS[
            "assessment_alignment"
        ],
        "Hiring Logistics": READINESS_WEIGHTS["hiring_logistics"],
    }

    for dimension in dimensions:
        expected_max_score = expected_max_scores[
            dimension.dimension_name
        ]

        if dimension.max_score != expected_max_score:
            raise RuntimeError(
                f"{dimension.dimension_name} max score mismatch: "
                f"expected {expected_max_score}, "
                f"found {dimension.max_score}."
            )

    total_score = sum(
        dimension.score
        for dimension in dimensions
    )

    if not 0 <= total_score <= 100:
        raise RuntimeError(
            f"Readiness score must be between 0 and 100. "
            f"Found {total_score}."
        )

    # --------------------------------------------------------
    # 8. DETERMINE READINESS STATUS
    # --------------------------------------------------------

    if blocking_gaps:
        readiness_status = ReadinessStatus.NOT_READY
        can_begin_sourcing = False

    elif total_score >= READINESS_THRESHOLDS["ready_to_source"]:
        readiness_status = ReadinessStatus.READY_TO_SOURCE
        can_begin_sourcing = True

    elif total_score >= READINESS_THRESHOLDS[
        "ready_with_assumptions"
    ]:
        readiness_status = ReadinessStatus.READY_WITH_ASSUMPTIONS
        can_begin_sourcing = True

    elif total_score >= READINESS_THRESHOLDS[
        "needs_clarification"
    ]:
        readiness_status = ReadinessStatus.NEEDS_CLARIFICATION
        can_begin_sourcing = False

    else:
        readiness_status = ReadinessStatus.NOT_READY
        can_begin_sourcing = False

    # --------------------------------------------------------
    # 9. IDENTIFY STRENGTHS
    # --------------------------------------------------------

    strengths = [
        f"{dimension.dimension_name} is well defined."
        for dimension in dimensions
        if (
            dimension.max_score > 0
            and dimension.score / dimension.max_score >= 0.75
        )
    ]

    # --------------------------------------------------------
    # 10. RECRUITER-FACING MESSAGE
    # --------------------------------------------------------

    if blocking_gaps:
        recruiter_message = (
            f"This role is not ready to source. "
            f"{len(blocking_gaps)} blocking decision(s) must be "
            "resolved with the hiring manager."
        )

    elif readiness_status == ReadinessStatus.READY_TO_SOURCE:
        recruiter_message = (
            "The role is sufficiently defined for recruiter review "
            "and approval."
        )

    elif readiness_status == ReadinessStatus.READY_WITH_ASSUMPTIONS:
        recruiter_message = (
            "Sourcing may begin if the remaining assumptions are "
            "documented and accepted by the recruiter."
        )

    elif readiness_status == ReadinessStatus.NEEDS_CLARIFICATION:
        recruiter_message = (
            "The role is partially defined but requires further "
            "clarification before sourcing begins."
        )

    else:
        recruiter_message = (
            "The role requires substantial clarification before a "
            "consistent sourcing search can be launched."
        )

    return ReadinessAssessment(
        total_score=total_score,
        readiness_status=readiness_status,
        dimensions=dimensions,
        blocking_gaps=blocking_gaps,
        high_impact_gaps=high_impact_gaps,
        helpful_gaps=helpful_gaps,
        strengths=strengths,
        recruiter_message=recruiter_message,
        can_begin_sourcing=can_begin_sourcing,
        requires_recruiter_approval=True,
    )


if not callable(calculate_sourcing_readiness):
    raise RuntimeError(
        "calculate_sourcing_readiness was not created successfully."
    )

print("✓ Deterministic Sourcing Readiness Tool created and verified")

✓ Deterministic Sourcing Readiness Tool created and verified


In [16]:
# ============================================================
# 5.6 ASSESS BRIEF V1 READINESS
# ============================================================

readiness_v1 = calculate_sourcing_readiness(brief_v1)

if not isinstance(readiness_v1, ReadinessAssessment):
    raise TypeError(
        "calculate_sourcing_readiness did not return "
        "a ReadinessAssessment."
    )

print("=" * 68)
print("S.O.N.A.R. SOURCING READINESS DIAGNOSIS")
print("=" * 68)

print(f"\nBrief version: {brief_v1.brief_version}")
print(f"Role: {brief_v1.role_title or 'Not confirmed'}")
print(f"Readiness Score: {readiness_v1.total_score}/100")
print(f"Readiness Status: {readiness_v1.readiness_status.value}")
print(f"Can begin sourcing: {readiness_v1.can_begin_sourcing}")

print("\nRecruiter Message")
print("-----------------")
print(readiness_v1.recruiter_message)

print("\nDimension Scores")
print("----------------")

for dimension in readiness_v1.dimensions:
    print(
        f"• {dimension.dimension_name}: "
        f"{dimension.score}/{dimension.max_score}"
    )

print("\nBlocking Gaps")
print("-------------")

if readiness_v1.blocking_gaps:
    for gap in readiness_v1.blocking_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHigh-Impact Gaps")
print("----------------")

if readiness_v1.high_impact_gaps:
    for gap in readiness_v1.high_impact_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHelpful Gaps")
print("------------")

if readiness_v1.helpful_gaps:
    for gap in readiness_v1.helpful_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nStrengths")
print("---------")

if readiness_v1.strengths:
    for strength in readiness_v1.strengths:
        print(f"• {strength}")
else:
    print("• No dimension is strongly complete yet.")

print("\nNext Step")
print("---------")
print(
    "Add recruiter intake notes or a hiring-manager transcript, "
    "then rerun the analyzer to create Brief v2."
)

S.O.N.A.R. SOURCING READINESS DIAGNOSIS

Brief version: 1
Role: Senior Data Engineer
Readiness Score: 28/100
Readiness Status: not_ready
Can begin sourcing: False

Recruiter Message
-----------------
This role is not ready to source. 3 blocking decision(s) must be resolved with the hiring manager.

Dimension Scores
----------------
• Role Definition: 17/20
• Candidate Eligibility: 5/25
• Search Parameters: 0/20
• Assessment Alignment: 6/20
• Hiring Logistics: 0/15

Blocking Gaps
-------------
• The mandatory skills are not sufficiently defined.
• The permitted hiring location has not been confirmed.
• The remote, hybrid, or on-site expectation has not been confirmed.

High-Impact Gaps
----------------
• It is unclear whether candidates at adjacent seniority levels may be considered.
• The hiring manager's non-negotiable requirements have not been explicitly documented.
• Acceptable substitutes for skills, technologies, or experience have not been clarified.
• Visa sponsorship availabil

# 6. Iterative Intake Update

Brief v1 was created from the job description only.

The next step adds recruiter intake notes or a hiring-manager transcript. This tests whether S.O.N.A.R. can update the brief when new context is provided.

This is where the workflow begins to become agentic:

```text
Brief v1
JD only
    ↓
Readiness Tool
Find gaps
    ↓
Add intake notes
    ↓
Brief v2
Updated sourcing brief
    ↓
Readiness Tool
Re-score and reassess

In [17]:
# ============================================================
# 6.1 CREATE BRIEF V2 FROM JD + INTAKE NOTES
# ============================================================

print("Analyzing job description with intake context...")

try:
    brief_v2, brief_v2_model = analyze_sourcing_brief(
        job_description=DEMO_CASE["job_description"],
        intake_context=DEMO_CASE["intake_context"],
    )

except Exception as error:
    raise RuntimeError(
        "Brief v2 could not be created from the JD and intake notes."
    ) from error


# ------------------------------------------------------------
# VERIFY THE RESULT
# ------------------------------------------------------------

if not isinstance(brief_v2, SourcingBrief):
    raise TypeError(
        "The analyzer did not return a valid SourcingBrief."
    )

if not brief_v2.role_title:
    raise RuntimeError(
        "The role title was not extracted in Brief v2."
    )

# Python controls versioning after extraction.
brief_v2.brief_version = 2


# ------------------------------------------------------------
# SUMMARY OUTPUT
# ------------------------------------------------------------

print("\n✓ Brief v2 created successfully")
print(f"✓ Model used: {brief_v2_model}")
print(f"✓ Role identified: {brief_v2.role_title}")
print(f"✓ Brief version: {brief_v2.brief_version}")
print(f"✓ Team: {brief_v2.team_name or 'Not confirmed'}")
print(
    "✓ Locations: "
    + (
        ", ".join(brief_v2.locations)
        if brief_v2.locations
        else "Not confirmed"
    )
)
print(f"✓ Work arrangement: {brief_v2.work_arrangement or 'Not confirmed'}")
print(f"✓ Evidence items captured: {len(brief_v2.source_evidence)}")
print(
    "✓ Clarification questions generated: "
    f"{len(brief_v2.clarification_questions)}"
)

Analyzing job description with intake context...

✓ Brief v2 created successfully
✓ Model used: gemini-2.5-flash-lite
✓ Role identified: Senior Data Engineer
✓ Brief version: 2
✓ Team: Data Platform
✓ Locations: San Francisco, Seattle
✓ Work arrangement: Hybrid
✓ Evidence items captured: 41
✓ Clarification questions generated: 3


In [18]:
# ============================================================
# 6.2 COMPARE BRIEF V1 AND BRIEF V2
# ============================================================

def format_value(value):
    """
    Format values for readable comparison output.
    """

    if is_missing(value):
        return "Not confirmed"

    if isinstance(value, list):
        return ", ".join(str(item) for item in value)

    return str(value)


COMPARISON_FIELDS = {
    "Team": "team_name",
    "Seniority": "seniority_levels",
    "Level Flexibility": "level_flexibility",
    "Locations": "locations",
    "Work Arrangement": "work_arrangement",
    "Number of Openings": "number_of_openings",
    "Target Hire Date": "target_hire_date",
    "Visa Sponsorship": "visa_sponsorship",
    "Compensation Range": "compensation_range",
    "Must-Have Skills": "must_have_skills",
    "Preferred Skills": "nice_to_have_skills",
    "Non-Negotiables": "non_negotiable_requirements",
    "Flexible Requirements": "flexible_requirements",
    "Acceptable Alternatives": "acceptable_alternatives",
    "Target Companies": "target_companies",
    "Screening Questions": "candidate_screening_questions",
    "Expected Screening Evidence": "expected_screening_evidence",
}

print("=" * 68)
print("S.O.N.A.R. BRIEF UPDATE SUMMARY")
print("=" * 68)

print("\nWhat changed after adding intake context?")
print("----------------------------------------")

changes_found = False

for label, field_name in COMPARISON_FIELDS.items():
    value_v1 = getattr(brief_v1, field_name)
    value_v2 = getattr(brief_v2, field_name)

    if value_v1 != value_v2:
        changes_found = True

        print(f"\n{label}")
        print(f"  Brief v1: {format_value(value_v1)}")
        print(f"  Brief v2: {format_value(value_v2)}")

if not changes_found:
    print("No material field changes detected.")

print("\nEvidence Growth")
print("---------------")
print(f"Brief v1 evidence items: {len(brief_v1.source_evidence)}")
print(f"Brief v2 evidence items: {len(brief_v2.source_evidence)}")

print("\nClarification Questions")
print("-----------------------")
print(f"Brief v1 questions: {len(brief_v1.clarification_questions)}")
print(f"Brief v2 questions: {len(brief_v2.clarification_questions)}")

S.O.N.A.R. BRIEF UPDATE SUMMARY

What changed after adding intake context?
----------------------------------------

Team
  Brief v1: Not confirmed
  Brief v2: Data Platform

Seniority
  Brief v1: Senior
  Brief v2: Senior, Mid-level

Level Flexibility
  Brief v1: Not confirmed
  Brief v2: The hiring manager prefers a senior candidate but may consider a strong mid-level candidate.

Locations
  Brief v1: Not confirmed
  Brief v2: San Francisco, Seattle

Work Arrangement
  Brief v1: Not confirmed
  Brief v2: Hybrid

Number of Openings
  Brief v1: Not confirmed
  Brief v2: 1

Must-Have Skills
  Brief v1: Not confirmed
  Brief v2: Python, SQL, Kafka

Preferred Skills
  Brief v1: Not confirmed
  Brief v2: Spark, Flink, Airflow, dbt

Non-Negotiables
  Brief v1: Not confirmed
  Brief v2: Python, SQL

Acceptable Alternatives
  Brief v1: Not confirmed
  Brief v2: Spark or Flink experience is acceptable

Target Companies
  Brief v1: Not confirmed
  Brief v2: Stripe, Uber, Airbnb, Block, Other hi

In [19]:
# ============================================================
# 6.3 ASSESS BRIEF V2 READINESS
# ============================================================

readiness_v2 = calculate_sourcing_readiness(brief_v2)

if not isinstance(readiness_v2, ReadinessAssessment):
    raise TypeError(
        "calculate_sourcing_readiness did not return "
        "a ReadinessAssessment."
    )

print("=" * 68)
print("S.O.N.A.R. UPDATED SOURCING READINESS DIAGNOSIS")
print("=" * 68)

print(f"\nBrief version: {brief_v2.brief_version}")
print(f"Role: {brief_v2.role_title or 'Not confirmed'}")
print(f"Readiness Score: {readiness_v2.total_score}/100")
print(f"Readiness Status: {readiness_v2.readiness_status.value}")
print(f"Can begin sourcing: {readiness_v2.can_begin_sourcing}")

print("\nScore Change")
print("------------")
print(f"Brief v1 score: {readiness_v1.total_score}/100")
print(f"Brief v2 score: {readiness_v2.total_score}/100")
print(
    f"Change: {readiness_v2.total_score - readiness_v1.total_score:+} points"
)

print("\nRecruiter Message")
print("-----------------")
print(readiness_v2.recruiter_message)

print("\nDimension Scores")
print("----------------")

for dimension in readiness_v2.dimensions:
    print(
        f"• {dimension.dimension_name}: "
        f"{dimension.score}/{dimension.max_score}"
    )

print("\nBlocking Gaps")
print("-------------")

if readiness_v2.blocking_gaps:
    for gap in readiness_v2.blocking_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHigh-Impact Gaps")
print("----------------")

if readiness_v2.high_impact_gaps:
    for gap in readiness_v2.high_impact_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHelpful Gaps")
print("------------")

if readiness_v2.helpful_gaps:
    for gap in readiness_v2.helpful_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nNext Step")
print("---------")
print(
    "Resolve the remaining high-impact gaps through recruiter or "
    "hiring-manager clarification, then create Brief v3."
)

S.O.N.A.R. UPDATED SOURCING READINESS DIAGNOSIS

Brief version: 2
Role: Senior Data Engineer
Readiness Score: 88/100
Readiness Status: ready_to_source
Can begin sourcing: True

Score Change
------------
Brief v1 score: 28/100
Brief v2 score: 88/100
Change: +60 points

Recruiter Message
-----------------
The role is sufficiently defined for recruiter review and approval.

Dimension Scores
----------------
• Role Definition: 20/20
• Candidate Eligibility: 25/25
• Search Parameters: 17/20
• Assessment Alignment: 20/20
• Hiring Logistics: 6/15

Blocking Gaps
-------------
• None

High-Impact Gaps
----------------
• Visa sponsorship availability has not been confirmed.
• The approved compensation range has not been provided.
• The target hiring date has not been confirmed.

Helpful Gaps
------------
• Target and adjacent job titles have not yet been defined.

Next Step
---------
Resolve the remaining high-impact gaps through recruiter or hiring-manager clarification, then create Brief v3.


# 7. Clarification Loop

Brief v2 is strong enough to begin sourcing, but it still has high-impact gaps:

- visa sponsorship is not confirmed;
- compensation range is not confirmed;
- target hiring date is not confirmed;
- target job titles are not documented.

The next step simulates recruiter or hiring-manager clarification.

This completes the core agentic loop:

```text
Analyze role
    ↓
Score readiness
    ↓
Identify unresolved gaps
    ↓
Ask targeted clarification questions
    ↓
Update structured memory
    ↓
Re-score
    ↓
Route to recruiter approval

In [20]:
# ============================================================
# 7.1 ADD CLARIFICATION ANSWERS
# ============================================================

DEMO_CASE["clarification_answers"] = """
Clarification answers from recruiter and hiring manager:

Visa sponsorship:
Yes, visa sponsorship is available for qualified candidates.

Compensation:
The approved compensation range is $170,000 to $210,000 base salary,
depending on level and experience.

Target hiring date:
The hiring manager wants the first qualified shortlist within
10 business days and would like to complete the hire within 45 days.

Target titles:
Relevant target titles include Senior Data Engineer, Data Platform
Engineer, Streaming Data Engineer, Backend Data Engineer, and
Analytics Infrastructure Engineer.

Employment type:
This is a full-time role.

Location flexibility:
Candidates should be based in San Francisco or Seattle. Nearby
commutable locations may be considered if the candidate can meet
the hybrid office expectation.
"""

if not DEMO_CASE["clarification_answers"].strip():
    raise RuntimeError(
        "Clarification answers were not added to the demo case."
    )

print("✓ Clarification answers added")
print("✓ Remaining high-impact gaps from Brief v2 are now addressed")
print("✓ Ready to create Brief v3")

✓ Clarification answers added
✓ Remaining high-impact gaps from Brief v2 are now addressed
✓ Ready to create Brief v3


In [21]:
# ============================================================
# 7.2 CREATE BRIEF V3 FROM CLARIFICATION ANSWERS
# ============================================================

print("Creating Brief v3 from JD, intake context, and clarification answers...")

try:
    brief_v3, brief_v3_model = analyze_sourcing_brief(
        job_description=DEMO_CASE["job_description"],
        intake_context=DEMO_CASE["intake_context"],
        clarification_answers=DEMO_CASE["clarification_answers"],
    )

except Exception as error:
    raise RuntimeError(
        "Brief v3 could not be created from the clarified recruiting context."
    ) from error


# ------------------------------------------------------------
# VERIFY THE RESULT
# ------------------------------------------------------------

if not isinstance(brief_v3, SourcingBrief):
    raise TypeError(
        "The analyzer did not return a valid SourcingBrief."
    )

if not brief_v3.role_title:
    raise RuntimeError(
        "The role title was not extracted in Brief v3."
    )

brief_v3.brief_version = 3


# ------------------------------------------------------------
# SUMMARY OUTPUT
# ------------------------------------------------------------

print("\n✓ Brief v3 created successfully")
print(f"✓ Model used: {brief_v3_model}")
print(f"✓ Role identified: {brief_v3.role_title}")
print(f"✓ Brief version: {brief_v3.brief_version}")
print(f"✓ Compensation: {brief_v3.compensation_range or 'Not confirmed'}")
print(
    "✓ Visa sponsorship: "
    + (
        str(brief_v3.visa_sponsorship)
        if brief_v3.visa_sponsorship is not None
        else "Not confirmed"
    )
)
print(f"✓ Target hire date: {brief_v3.target_hire_date or 'Not confirmed'}")
print(f"✓ Employment type: {brief_v3.employment_type or 'Not confirmed'}")
print(
    "✓ Target titles: "
    + (
        ", ".join(brief_v3.target_job_titles)
        if brief_v3.target_job_titles
        else "Not confirmed"
    )
)
print(f"✓ Evidence items captured: {len(brief_v3.source_evidence)}")
print(
    "✓ Clarification questions generated: "
    f"{len(brief_v3.clarification_questions)}"
)

Creating Brief v3 from JD, intake context, and clarification answers...
Temporary issue with gemini-2.5-flash-lite. Retrying in 2.2 seconds...
Temporary issue with gemini-2.5-flash-lite. Retrying in 4.9 seconds...
gemini-2.5-flash-lite remained unavailable. Trying the next configured model...

✓ Brief v3 created successfully
✓ Model used: gemini-2.5-flash
✓ Role identified: Senior Data Engineer
✓ Brief version: 3
✓ Compensation: $170,000 to $210,000 base salary
✓ Visa sponsorship: True
✓ Target hire date: complete the hire within 45 days
✓ Employment type: full-time
✓ Target titles: Senior Data Engineer, Data Platform Engineer, Streaming Data Engineer, Backend Data Engineer, Analytics Infrastructure Engineer
✓ Evidence items captured: 43
✓ Clarification questions generated: 3


In [22]:
# ============================================================
# 7.3 ASSESS BRIEF V3 READINESS
# ============================================================

readiness_v3 = calculate_sourcing_readiness(brief_v3)

if not isinstance(readiness_v3, ReadinessAssessment):
    raise TypeError(
        "calculate_sourcing_readiness did not return "
        "a ReadinessAssessment."
    )

print("=" * 68)
print("S.O.N.A.R. FINAL CLARIFIED READINESS DIAGNOSIS")
print("=" * 68)

print(f"\nBrief version: {brief_v3.brief_version}")
print(f"Role: {brief_v3.role_title or 'Not confirmed'}")
print(f"Readiness Score: {readiness_v3.total_score}/100")
print(f"Readiness Status: {readiness_v3.readiness_status.value}")
print(f"Can begin sourcing: {readiness_v3.can_begin_sourcing}")

print("\nScore Progression")
print("-----------------")
print(f"Brief v1 score: {readiness_v1.total_score}/100")
print(f"Brief v2 score: {readiness_v2.total_score}/100")
print(f"Brief v3 score: {readiness_v3.total_score}/100")
print(
    f"Total improvement: "
    f"{readiness_v3.total_score - readiness_v1.total_score:+} points"
)

print("\nRecruiter Message")
print("-----------------")
print(readiness_v3.recruiter_message)

print("\nDimension Scores")
print("----------------")

for dimension in readiness_v3.dimensions:
    print(
        f"• {dimension.dimension_name}: "
        f"{dimension.score}/{dimension.max_score}"
    )

print("\nBlocking Gaps")
print("-------------")

if readiness_v3.blocking_gaps:
    for gap in readiness_v3.blocking_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHigh-Impact Gaps")
print("----------------")

if readiness_v3.high_impact_gaps:
    for gap in readiness_v3.high_impact_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nHelpful Gaps")
print("------------")

if readiness_v3.helpful_gaps:
    for gap in readiness_v3.helpful_gaps:
        print(f"• {gap}")
else:
    print("• None")

print("\nClarification Questions Still Suggested")
print("---------------------------------------")

if brief_v3.clarification_questions:
    for index, question in enumerate(
        brief_v3.clarification_questions,
        start=1,
    ):
        print(
            f"{index}. [{question.priority.value.upper()}] "
            f"{question.question}"
        )
else:
    print("No further clarification questions were generated.")

print("\nAgent Loop Status")
print("-----------------")

if readiness_v3.can_begin_sourcing:
    print(
        "The role is ready to move to the recruiter approval gate."
    )
else:
    print(
        "The role still requires clarification before recruiter approval."
    )

S.O.N.A.R. FINAL CLARIFIED READINESS DIAGNOSIS

Brief version: 3
Role: Senior Data Engineer
Readiness Score: 93/100
Readiness Status: ready_to_source
Can begin sourcing: True

Score Progression
-----------------
Brief v1 score: 28/100
Brief v2 score: 88/100
Brief v3 score: 93/100
Total improvement: +65 points

Recruiter Message
-----------------
The role is sufficiently defined for recruiter review and approval.

Dimension Scores
----------------
• Role Definition: 20/20
• Candidate Eligibility: 25/25
• Search Parameters: 20/20
• Assessment Alignment: 13/20
• Hiring Logistics: 15/15

Blocking Gaps
-------------
• None

High-Impact Gaps
----------------
• Unresolved ambiguity: The specific distributed data-processing systems and cloud infrastructure platforms that are considered must-haves are not detailed.
• Unresolved ambiguity: The definition of 'high-scale technology environment' is broad.

Helpful Gaps
------------
• Candidate screening questions have not yet been documented.

Clar

# 8. Agent Routing and Human Approval

S.O.N.A.R. does not automatically launch sourcing.

The agent can recommend the next workflow step, but the recruiter remains the final decision-maker. This is important for enterprise safety, accountability, and human oversight.

The Approval Router uses the deterministic readiness assessment to decide whether the role should move to:

1. more clarification;
2. recruiter review;
3. approved sourcing launch;
4. revision.

This section demonstrates human-in-the-loop agent design.

In [23]:
# ============================================================
# 8.1 AGENT ROUTING DECISION
# ============================================================

class AgentAction(str, Enum):
    REQUEST_BLOCKING_CLARIFICATION = "request_blocking_clarification"
    REQUEST_HIGH_IMPACT_CLARIFICATION = "request_high_impact_clarification"
    REQUEST_RECRUITER_APPROVAL = "request_recruiter_approval"
    APPROVED_FOR_SOURCING = "approved_for_sourcing"
    REVISE_ROLE_BRIEF = "revise_role_brief"


class AgentRoutingDecision(BaseModel):
    """
    Router output that determines the next action in the workflow.
    """

    recommended_action: AgentAction
    reason: str
    next_owner: str
    required_inputs: List[str] = Field(default_factory=list)
    can_proceed_without_additional_llm_call: bool


def route_next_action(
    brief: SourcingBrief,
    readiness: ReadinessAssessment
) -> AgentRoutingDecision:
    """
    Decide the next workflow step based on readiness status,
    unresolved gaps, and recruiter approval state.
    """

    if not isinstance(brief, SourcingBrief):
        raise TypeError("brief must be a SourcingBrief.")

    if not isinstance(readiness, ReadinessAssessment):
        raise TypeError("readiness must be a ReadinessAssessment.")

    if readiness.blocking_gaps:
        return AgentRoutingDecision(
            recommended_action=(
                AgentAction.REQUEST_BLOCKING_CLARIFICATION
            ),
            reason=(
                "The role has blocking gaps that could prevent a "
                "consistent sourcing search."
            ),
            next_owner="Recruiter or hiring manager",
            required_inputs=readiness.blocking_gaps,
            can_proceed_without_additional_llm_call=False,
        )

    if (
        not readiness.can_begin_sourcing
        and readiness.high_impact_gaps
    ):
        return AgentRoutingDecision(
            recommended_action=(
                AgentAction.REQUEST_HIGH_IMPACT_CLARIFICATION
            ),
            reason=(
                "The role is partially defined, but high-impact gaps "
                "could materially affect the candidate pool or search "
                "strategy."
            ),
            next_owner="Recruiter or hiring manager",
            required_inputs=readiness.high_impact_gaps,
            can_proceed_without_additional_llm_call=False,
        )

    if brief.approval_status == ApprovalStatus.APPROVED:
        return AgentRoutingDecision(
            recommended_action=AgentAction.APPROVED_FOR_SOURCING,
            reason=(
                "The role is ready and has been approved by the recruiter."
            ),
            next_owner="Recruiting team",
            required_inputs=[],
            can_proceed_without_additional_llm_call=True,
        )

    if readiness.can_begin_sourcing:
        return AgentRoutingDecision(
            recommended_action=AgentAction.REQUEST_RECRUITER_APPROVAL,
            reason=(
                "The role is sufficiently defined for sourcing, but "
                "human approval is required before launch."
            ),
            next_owner="Recruiter",
            required_inputs=[
                "Recruiter approval",
                "Acceptance of any remaining assumptions",
            ],
            can_proceed_without_additional_llm_call=True,
        )

    return AgentRoutingDecision(
        recommended_action=AgentAction.REVISE_ROLE_BRIEF,
        reason=(
            "The role is not ready and requires further revision before "
            "sourcing can begin."
        ),
        next_owner="Recruiter",
        required_inputs=[
            "Updated job description",
            "Additional hiring-manager context",
        ],
        can_proceed_without_additional_llm_call=False,
    )


routing_v3 = route_next_action(
    brief=brief_v3,
    readiness=readiness_v3,
)

print("=" * 68)
print("S.O.N.A.R. APPROVAL ROUTER")
print("=" * 68)

print(f"\nRecommended action: {routing_v3.recommended_action.value}")
print(f"Next owner: {routing_v3.next_owner}")
print(f"Reason: {routing_v3.reason}")
print(
    "Can proceed without another LLM call: "
    f"{routing_v3.can_proceed_without_additional_llm_call}"
)

print("\nRequired Inputs")
print("---------------")

if routing_v3.required_inputs:
    for item in routing_v3.required_inputs:
        print(f"• {item}")
else:
    print("• None")

S.O.N.A.R. APPROVAL ROUTER

Recommended action: request_recruiter_approval
Next owner: Recruiter
Reason: The role is sufficiently defined for sourcing, but human approval is required before launch.
Can proceed without another LLM call: True

Required Inputs
---------------
• Recruiter approval
• Acceptance of any remaining assumptions


In [24]:
# ============================================================
# 8.2 RECRUITER APPROVAL GATE
# ============================================================

def apply_recruiter_approval(
    brief: SourcingBrief,
    readiness: ReadinessAssessment,
    recruiter_approves: bool,
    approval_note: str = ""
) -> SourcingBrief:
    """
    Apply human-in-the-loop recruiter approval.

    The agent can recommend approval, but only the recruiter can
    approve the role for sourcing launch.
    """

    if not isinstance(brief, SourcingBrief):
        raise TypeError("brief must be a SourcingBrief.")

    if not isinstance(readiness, ReadinessAssessment):
        raise TypeError("readiness must be a ReadinessAssessment.")

    updated_brief = brief.model_copy(deep=True)

    if recruiter_approves and readiness.can_begin_sourcing:
        updated_brief.approval_status = ApprovalStatus.APPROVED

        if approval_note.strip():
            updated_brief.sourcing_caveats.append(
                f"Recruiter approval note: {approval_note.strip()}"
            )

    elif recruiter_approves and not readiness.can_begin_sourcing:
        updated_brief.approval_status = (
            ApprovalStatus.REJECTED_FOR_REVISION
        )
        updated_brief.sourcing_caveats.append(
            "Recruiter attempted approval, but the role is not ready "
            "according to the readiness tool."
        )

    else:
        updated_brief.approval_status = (
            ApprovalStatus.REJECTED_FOR_REVISION
        )

        if approval_note.strip():
            updated_brief.sourcing_caveats.append(
                f"Recruiter revision note: {approval_note.strip()}"
            )

    return updated_brief


# Simulated recruiter decision for the controlled demo.
# In Streamlit, this will become an approval checkbox or button.
RECRUITER_APPROVES_DEMO = True

RECRUITER_APPROVAL_NOTE = """
Approved to begin sourcing. Recruiter accepts the remaining helpful
questions as non-blocking and will clarify them during the first
sourcing calibration.
"""

approved_brief = apply_recruiter_approval(
    brief=brief_v3,
    readiness=readiness_v3,
    recruiter_approves=RECRUITER_APPROVES_DEMO,
    approval_note=RECRUITER_APPROVAL_NOTE,
)

routing_after_approval = route_next_action(
    brief=approved_brief,
    readiness=readiness_v3,
)

print("=" * 68)
print("S.O.N.A.R. HUMAN APPROVAL GATE")
print("=" * 68)

print(f"\nRecruiter approved: {RECRUITER_APPROVES_DEMO}")
print(f"Approval status: {approved_brief.approval_status.value}")
print(
    "Post-approval routing action: "
    f"{routing_after_approval.recommended_action.value}"
)
print(f"Next owner: {routing_after_approval.next_owner}")
print(f"Reason: {routing_after_approval.reason}")

print("\nApproval Notes and Caveats")
print("--------------------------")

if approved_brief.sourcing_caveats:
    for caveat in approved_brief.sourcing_caveats:
        print(f"• {caveat}")
else:
    print("• None")

S.O.N.A.R. HUMAN APPROVAL GATE

Recruiter approved: True
Approval status: approved
Post-approval routing action: approved_for_sourcing
Next owner: Recruiting team
Reason: The role is ready and has been approved by the recruiter.

Approval Notes and Caveats
--------------------------
• Recruiter approval note: Approved to begin sourcing. Recruiter accepts the remaining helpful
questions as non-blocking and will clarify them during the first
sourcing calibration.


# 9. Sourcing Launch Readiness Plan

After recruiter approval, S.O.N.A.R. creates a lightweight sourcing launch plan.

This section uses a deterministic rule-based tool. It is not a trained forecasting model and does not claim to predict hiring outcomes.

The goal is to give recruiters a transparent starting point for launch planning:

- how difficult the search appears;
- what factors drive that difficulty;
- what the first sourcing focus should be;
- what assumptions remain;
- what would need calibration before production use.

In production, these rules should be calibrated using historical recruiting data such as:

- time to first qualified shortlist;
- candidate response rates;
- hiring-manager rejection reasons;
- role complexity;
- compensation competitiveness;
- location constraints.

In [26]:
# ============================================================
# 9.1 RULE-BASED SOURCING LAUNCH READINESS PLAN
# ============================================================

class SourcingLaunchPlan(BaseModel):
    """
    Rule-based launch plan created after recruiter approval.

    This is a prototype planning aid, not a validated predictive model.
    """

    role_title: str
    approval_status: ApprovalStatus
    difficulty_band: str
    prototype_shortlist_estimate: str
    difficulty_factors: List[str] = Field(default_factory=list)
    recommended_initial_search_focus: List[str] = Field(default_factory=list)
    remaining_assumptions: List[str] = Field(default_factory=list)
    calibration_note: str
    launch_rationale: str


def estimate_sourcing_launch_plan(
    brief: SourcingBrief,
    readiness: ReadinessAssessment
) -> SourcingLaunchPlan:
    """
    Create a transparent, rule-based sourcing launch plan.

    This tool is intentionally deterministic and explainable.
    It should be calibrated with historical recruiting data before
    being used as a production forecasting tool.
    """

    if not isinstance(brief, SourcingBrief):
        raise TypeError("brief must be a SourcingBrief.")

    if not isinstance(readiness, ReadinessAssessment):
        raise TypeError("readiness must be a ReadinessAssessment.")

    if brief.approval_status != ApprovalStatus.APPROVED:
        raise RuntimeError(
            "A sourcing launch plan can only be created after "
            "recruiter approval."
        )

    difficulty_points = 0
    difficulty_factors = []

    seniority_text = " ".join(brief.seniority_levels).lower()

    if "senior" in seniority_text:
        difficulty_points += 2
        difficulty_factors.append(
            "Senior-level roles usually require a narrower candidate pool."
        )

    must_have_skills_text = " ".join(brief.must_have_skills).lower()

    if "kafka" in must_have_skills_text:
        difficulty_points += 2
        difficulty_factors.append(
            "Kafka is a mandatory skill, which narrows the sourcing pool."
        )

    if brief.locations and len(brief.locations) <= 2:
        difficulty_points += 1
        difficulty_factors.append(
            "The search is limited to a small number of approved locations."
        )

    if brief.compensation_range:
        difficulty_factors.append(
            "Compensation is confirmed, which reduces launch uncertainty."
        )
    else:
        difficulty_points += 2
        difficulty_factors.append(
            "Compensation is unconfirmed, which increases launch uncertainty."
        )

    if brief.visa_sponsorship is True:
        difficulty_points -= 1
        difficulty_factors.append(
            "Visa sponsorship is available, which may expand the candidate pool."
        )

    if readiness.high_impact_gaps:
        difficulty_points += 1
        difficulty_factors.append(
            "Some high-impact ambiguity remains and should be monitored."
        )

    if difficulty_points <= 2:
        difficulty_band = "moderate"
        prototype_shortlist_estimate = "5 to 7 business days"

    elif difficulty_points <= 4:
        difficulty_band = "moderate-high"
        prototype_shortlist_estimate = "7 to 10 business days"

    else:
        difficulty_band = "high"
        prototype_shortlist_estimate = "10 to 15 business days"

    recommended_focus = []

    if brief.target_job_titles:
        recommended_focus.extend(
            f"Search for candidates with title: {title}"
            for title in brief.target_job_titles[:5]
        )

    if brief.target_companies:
        recommended_focus.append(
            "Prioritize high-scale technology companies such as "
            + ", ".join(brief.target_companies[:4])
            + "."
        )

    if brief.must_have_skills:
        recommended_focus.append(
            "Screen early for must-have skills: "
            + ", ".join(brief.must_have_skills)
            + "."
        )

    remaining_assumptions = []

    if readiness.high_impact_gaps:
        remaining_assumptions.extend(readiness.high_impact_gaps)

    if readiness.helpful_gaps:
        remaining_assumptions.extend(readiness.helpful_gaps)

    calibration_note = (
        "This is a prototype rule-based estimate. It is not a trained "
        "forecasting model. In production, the thresholds and time bands "
        "should be calibrated using company-specific recruiting data."
    )

    launch_rationale = (
        "The role has passed the readiness check and has recruiter approval. "
        "The launch plan uses transparent business rules based on seniority, "
        "mandatory skills, location constraints, compensation clarity, "
        "sponsorship availability, and unresolved assumptions."
    )

    return SourcingLaunchPlan(
        role_title=brief.role_title or "Unconfirmed role",
        approval_status=brief.approval_status,
        difficulty_band=difficulty_band,
        prototype_shortlist_estimate=prototype_shortlist_estimate,
        difficulty_factors=difficulty_factors,
        recommended_initial_search_focus=recommended_focus,
        remaining_assumptions=remaining_assumptions,
        calibration_note=calibration_note,
        launch_rationale=launch_rationale,
    )


launch_plan = estimate_sourcing_launch_plan(
    brief=approved_brief,
    readiness=readiness_v3,
)

print("=" * 68)
print("S.O.N.A.R. SOURCING LAUNCH READINESS PLAN")
print("=" * 68)

print(f"\nRole: {launch_plan.role_title}")
print(f"Approval status: {launch_plan.approval_status.value}")
print(f"Difficulty band: {launch_plan.difficulty_band}")
print(
    "Prototype shortlist estimate: "
    f"{launch_plan.prototype_shortlist_estimate}"
)

print("\nDifficulty Factors")
print("------------------")

if launch_plan.difficulty_factors:
    for factor in launch_plan.difficulty_factors:
        print(f"• {factor}")
else:
    print("• No major difficulty factors identified.")

print("\nRecommended Initial Search Focus")
print("--------------------------------")

if launch_plan.recommended_initial_search_focus:
    for item in launch_plan.recommended_initial_search_focus:
        print(f"• {item}")
else:
    print("• No search focus was generated.")

print("\nRemaining Assumptions")
print("---------------------")

if launch_plan.remaining_assumptions:
    for assumption in launch_plan.remaining_assumptions:
        print(f"• {assumption}")
else:
    print("• None")

print("\nLaunch Rationale")
print("----------------")
print(launch_plan.launch_rationale)

print("\nCalibration Note")
print("----------------")
print(launch_plan.calibration_note)

S.O.N.A.R. SOURCING LAUNCH READINESS PLAN

Role: Senior Data Engineer
Approval status: approved
Difficulty band: high
Prototype shortlist estimate: 10 to 15 business days

Difficulty Factors
------------------
• Senior-level roles usually require a narrower candidate pool.
• Kafka is a mandatory skill, which narrows the sourcing pool.
• The search is limited to a small number of approved locations.
• Compensation is confirmed, which reduces launch uncertainty.
• Visa sponsorship is available, which may expand the candidate pool.
• Some high-impact ambiguity remains and should be monitored.

Recommended Initial Search Focus
--------------------------------
• Search for candidates with title: Senior Data Engineer
• Search for candidates with title: Data Platform Engineer
• Search for candidates with title: Streaming Data Engineer
• Search for candidates with title: Backend Data Engineer
• Search for candidates with title: Analytics Infrastructure Engineer
• Prioritize high-scale technolo

# 9.2 Future Statistical Calibration Plan

The sourcing launch estimate in this prototype is intentionally rule-based and explainable.

It should not be interpreted as a statistically validated forecast. The demo does not include historical recruiting outcome data, so S.O.N.A.R. does not generate confidence intervals or claim predictive accuracy.

In production, this module could be calibrated using historical recruiting data such as:

- time to first qualified shortlist;
- time to hire;
- candidate response rates;
- qualified-candidate rate;
- hiring-manager rejection reasons;
- compensation competitiveness;
- location constraints;
- seniority and skill scarcity.

Potential statistical extensions include:

- regression or survival analysis for time-to-shortlist estimation;
- calibrated classification for predicting whether a role is ready to source;
- bootstrapped prediction intervals for uncertainty;
- precision, recall, and F1 evaluation for readiness decisions;
- periodic recalibration by role family, market, and geography.

For the hackathon MVP, the priority is transparent agent behavior: Gemini extracts structured evidence, deterministic tools apply business rules, and the recruiter retains final approval.

# 10. Evaluation and Reliability Checks

Agentic workflows need evaluation because the agent may fail in different ways.

In this notebook, S.O.N.A.R. surfaces three important reliability issues:

1. **Temporary model/API failure**  
   The primary Gemini model may be unavailable or rate-limited. S.O.N.A.R. uses retry and fallback logic instead of failing silently.

2. **State regression across iterations**  
   A later brief version can accidentally omit useful information captured in an earlier version. For example, Brief v2 captured screening questions, while Brief v3 did not preserve them consistently. The deterministic readiness tool caught this as a remaining helpful gap.

3. **Unsupported or unsafe extraction**  
   The agent should not invent requirements or use protected personal characteristics as sourcing criteria.

The goal of this evaluation harness is not to prove production accuracy. It is to make the agent's behavior inspectable, testable, and safer to improve.

In [27]:
# ============================================================
# 10.1 MODEL RESILIENCE AND STATE REGRESSION EVALUATION
# ============================================================

class ReliabilityCheck(BaseModel):
    """
    Evaluation result for reliability and state consistency checks.
    """

    check_name: str
    status: str
    observation: str
    implication: str
    mitigation: str


def evaluate_reliability_and_state_regression(
    brief_v1: SourcingBrief,
    brief_v2: SourcingBrief,
    brief_v3: SourcingBrief,
    model_v1: str,
    model_v2: str,
    model_v3: str,
) -> List[ReliabilityCheck]:
    """
    Evaluate model resilience and whether later brief versions
    preserved important information from earlier versions.
    """

    checks = []

    # --------------------------------------------------------
    # Model fallback / resilience check
    # --------------------------------------------------------

    fallback_used = model_v3 != model_v1 or model_v3 != model_v2

    checks.append(
        ReliabilityCheck(
            check_name="Model fallback behavior",
            status="PASS" if model_v3 else "REVIEW",
            observation=(
                f"Brief v1 used {model_v1}; "
                f"Brief v2 used {model_v2}; "
                f"Brief v3 used {model_v3}."
            ),
            implication=(
                "The workflow can continue when the primary model has "
                "temporary availability issues."
                if fallback_used
                else "The same model handled all brief versions."
            ),
            mitigation=(
                "Retry and fallback logic is implemented in "
                "analyze_sourcing_brief()."
            ),
        )
    )

    # --------------------------------------------------------
    # Screening-question regression check
    # --------------------------------------------------------

    v2_screening_count = len(brief_v2.candidate_screening_questions)
    v3_screening_count = len(brief_v3.candidate_screening_questions)

    screening_regression = (
        v2_screening_count > 0
        and v3_screening_count == 0
    )

    checks.append(
        ReliabilityCheck(
            check_name="Screening question preservation",
            status="REVIEW" if screening_regression else "PASS",
            observation=(
                f"Brief v2 screening questions: {v2_screening_count}; "
                f"Brief v3 screening questions: {v3_screening_count}."
            ),
            implication=(
                "A later LLM response dropped useful screening information "
                "captured in an earlier version."
                if screening_regression
                else "Screening information was preserved or was not "
                     "available in earlier versions."
            ),
            mitigation=(
                "Production version should merge useful prior-state fields "
                "instead of replacing the entire brief on each iteration."
            ),
        )
    )

    # --------------------------------------------------------
    # Evidence growth check
    # --------------------------------------------------------

    evidence_growth_ok = (
        len(brief_v3.source_evidence)
        >= len(brief_v1.source_evidence)
    )

    checks.append(
        ReliabilityCheck(
            check_name="Evidence traceability growth",
            status="PASS" if evidence_growth_ok else "REVIEW",
            observation=(
                f"Brief v1 evidence items: {len(brief_v1.source_evidence)}; "
                f"Brief v2 evidence items: {len(brief_v2.source_evidence)}; "
                f"Brief v3 evidence items: {len(brief_v3.source_evidence)}."
            ),
            implication=(
                "The agent maintained or expanded traceability as more "
                "context was added."
                if evidence_growth_ok
                else "Evidence traceability decreased and should be reviewed."
            ),
            mitigation=(
                "Keep evidence IDs and source excerpts for important "
                "extracted facts."
            ),
        )
    )

    # --------------------------------------------------------
    # Readiness progression check
    # --------------------------------------------------------

    score_progression_ok = (
        readiness_v1.total_score
        <= readiness_v2.total_score
        <= readiness_v3.total_score
    )

    checks.append(
        ReliabilityCheck(
            check_name="Readiness progression",
            status="PASS" if score_progression_ok else "REVIEW",
            observation=(
                f"Readiness scores: "
                f"v1={readiness_v1.total_score}, "
                f"v2={readiness_v2.total_score}, "
                f"v3={readiness_v3.total_score}."
            ),
            implication=(
                "The agent loop improved the role's sourcing readiness "
                "as new context was added."
                if score_progression_ok
                else "Readiness did not improve monotonically and should "
                     "be inspected."
            ),
            mitigation=(
                "Use deterministic readiness scoring to make iteration "
                "effects visible."
            ),
        )
    )

    return checks


reliability_checks = evaluate_reliability_and_state_regression(
    brief_v1=brief_v1,
    brief_v2=brief_v2,
    brief_v3=brief_v3,
    model_v1=brief_v1_model,
    model_v2=brief_v2_model,
    model_v3=brief_v3_model,
)

print("=" * 68)
print("S.O.N.A.R. RELIABILITY AND STATE REGRESSION EVALUATION")
print("=" * 68)

for check in reliability_checks:
    print(f"\n[{check.status}] {check.check_name}")
    print(f"Observation: {check.observation}")
    print(f"Implication: {check.implication}")
    print(f"Mitigation: {check.mitigation}")

S.O.N.A.R. RELIABILITY AND STATE REGRESSION EVALUATION

[PASS] Model fallback behavior
Observation: Brief v1 used gemini-2.5-flash-lite; Brief v2 used gemini-2.5-flash-lite; Brief v3 used gemini-2.5-flash.
Implication: The workflow can continue when the primary model has temporary availability issues.
Mitigation: Retry and fallback logic is implemented in analyze_sourcing_brief().

[REVIEW] Screening question preservation
Observation: Brief v2 screening questions: 1; Brief v3 screening questions: 0.
Implication: A later LLM response dropped useful screening information captured in an earlier version.
Mitigation: Production version should merge useful prior-state fields instead of replacing the entire brief on each iteration.

[PASS] Evidence traceability growth
Observation: Brief v1 evidence items: 16; Brief v2 evidence items: 41; Brief v3 evidence items: 43.
Implication: The agent maintained or expanded traceability as more context was added.
Mitigation: Keep evidence IDs and source e

In [28]:
# ============================================================
# 10.2 EXPECTED SIGNAL EVALUATION
# ============================================================

class EvaluationResult(BaseModel):
    """
    Result of one evaluation check.
    """

    check_name: str
    status: str
    expected: str
    observed: str
    explanation: str


def normalize_text(value: str) -> str:
    """
    Normalize text for simple evaluation comparisons.
    """

    return value.lower().strip()


def list_contains_signal(items: List[str], signal: str) -> bool:
    """
    Check whether a list contains an expected signal.
    """

    normalized_signal = normalize_text(signal)

    return any(
        normalized_signal in normalize_text(str(item))
        for item in items
    )


def evaluate_expected_signals(
    brief: SourcingBrief,
    readiness: ReadinessAssessment,
    expected_signals: dict
) -> List[EvaluationResult]:
    """
    Evaluate whether the final sourcing brief captured expected signals
    from the controlled demonstration case.
    """

    results = []

    # --------------------------------------------------------
    # Must-have skills
    # --------------------------------------------------------

    for skill in expected_signals["must_have_skills"]:
        observed = ", ".join(brief.must_have_skills)

        passed = list_contains_signal(
            brief.must_have_skills,
            skill,
        )

        results.append(
            EvaluationResult(
                check_name=f"Must-have skill captured: {skill}",
                status="PASS" if passed else "REVIEW",
                expected=skill,
                observed=observed or "No must-have skills found",
                explanation=(
                    "Checks whether the final brief captured an expected "
                    "must-have skill."
                ),
            )
        )

    # --------------------------------------------------------
    # Acceptable alternatives
    # --------------------------------------------------------

    for alternative in expected_signals["acceptable_alternatives"]:
        observed = ", ".join(brief.acceptable_alternatives)

        passed = list_contains_signal(
            brief.acceptable_alternatives,
            alternative,
        )

        results.append(
            EvaluationResult(
                check_name=(
                    f"Acceptable alternative captured: {alternative}"
                ),
                status="PASS" if passed else "REVIEW",
                expected=alternative,
                observed=(
                    observed
                    or "No acceptable alternatives found"
                ),
                explanation=(
                    "Checks whether the final brief captured a hiring-manager "
                    "approved substitute or flexibility rule."
                ),
            )
        )

    # --------------------------------------------------------
    # Locations
    # --------------------------------------------------------

    for location in expected_signals["locations"]:
        observed = ", ".join(brief.locations)

        passed = list_contains_signal(
            brief.locations,
            location,
        )

        results.append(
            EvaluationResult(
                check_name=f"Location captured: {location}",
                status="PASS" if passed else "REVIEW",
                expected=location,
                observed=observed or "No locations found",
                explanation=(
                    "Checks whether the final brief captured an approved "
                    "sourcing location."
                ),
            )
        )

    # --------------------------------------------------------
    # Previously unresolved topics
    # --------------------------------------------------------

    unresolved_text = " ".join(
        [
            *brief.missing_information,
            *brief.ambiguities,
            *readiness.blocking_gaps,
            *readiness.high_impact_gaps,
            *readiness.helpful_gaps,
        ]
    )

    for topic in expected_signals["unresolved_topics"]:
        passed = topic.lower() not in unresolved_text.lower()

        results.append(
            EvaluationResult(
                check_name=f"Clarified topic resolved: {topic}",
                status="PASS" if passed else "REVIEW",
                expected=(
                    f"{topic} should no longer be treated as unresolved"
                ),
                observed=unresolved_text or "No unresolved text found",
                explanation=(
                    "Checks that topics answered during clarification are "
                    "not still treated as unresolved gaps."
                ),
            )
        )

    return results


signal_eval_results = evaluate_expected_signals(
    brief=brief_v3,
    readiness=readiness_v3,
    expected_signals=DEMO_CASE["expected_signals"],
)

passed_signal_checks = sum(
    result.status == "PASS"
    for result in signal_eval_results
)

total_signal_checks = len(signal_eval_results)

print("=" * 68)
print("S.O.N.A.R. EXPECTED SIGNAL EVALUATION")
print("=" * 68)

print(f"\nPassed checks: {passed_signal_checks}/{total_signal_checks}")

for result in signal_eval_results:
    print(f"\n[{result.status}] {result.check_name}")
    print(f"Expected: {result.expected}")
    print(f"Observed: {result.observed}")
    print(f"Explanation: {result.explanation}")

if passed_signal_checks < total_signal_checks:
    print(
        "\nNote: REVIEW does not automatically mean failure. "
        "It identifies cases that should be inspected before production use."
    )
else:
    print(
        "\nAll expected controlled-case signals were captured or resolved."
    )

S.O.N.A.R. EXPECTED SIGNAL EVALUATION

Passed checks: 9/9

[PASS] Must-have skill captured: Python
Expected: Python
Observed: Python, SQL, Distributed data-processing systems, Cloud infrastructure, Kafka
Explanation: Checks whether the final brief captured an expected must-have skill.

[PASS] Must-have skill captured: SQL
Expected: SQL
Observed: Python, SQL, Distributed data-processing systems, Cloud infrastructure, Kafka
Explanation: Checks whether the final brief captured an expected must-have skill.

[PASS] Must-have skill captured: Kafka
Expected: Kafka
Observed: Python, SQL, Distributed data-processing systems, Cloud infrastructure, Kafka
Explanation: Checks whether the final brief captured an expected must-have skill.

[PASS] Acceptable alternative captured: Spark or Flink
Expected: Spark or Flink
Observed: Spark or Flink experience
Explanation: Checks whether the final brief captured a hiring-manager approved substitute or flexibility rule.

[PASS] Location captured: San Francis

In [29]:
# ============================================================
# 10.3 SECURITY AND SAFETY EVALUATION
# ============================================================

PROTECTED_OR_SENSITIVE_TERMS = [
    "age",
    "gender",
    "race",
    "ethnicity",
    "religion",
    "nationality",
    "marital status",
    "disability",
    "pregnancy",
    "sexual orientation",
    "citizenship",
]


class SafetyCheck(BaseModel):
    """
    Result of one security or safety check.
    """

    check_name: str
    status: str
    observation: str
    mitigation: str


def evaluate_security_and_safety(
    brief: SourcingBrief
) -> List[SafetyCheck]:
    """
    Check whether the generated sourcing brief avoids unsafe or
    unsupported sourcing behavior.
    """

    checks = []

    searchable_brief_text = " ".join(
        [
            brief.role_title or "",
            brief.team_name or "",
            " ".join(brief.must_have_skills),
            " ".join(brief.nice_to_have_skills),
            " ".join(brief.must_have_qualifications),
            " ".join(brief.nice_to_have_qualifications),
            " ".join(brief.non_negotiable_requirements),
            " ".join(brief.flexible_requirements),
            " ".join(brief.acceptable_alternatives),
            " ".join(brief.disqualifying_conditions),
            " ".join(brief.target_companies),
            " ".join(brief.target_job_titles),
            " ".join(brief.adjacent_profile_types),
            " ".join(brief.candidate_screening_questions),
            " ".join(brief.expected_screening_evidence),
        ]
    ).lower()

    detected_terms = [
        term
        for term in PROTECTED_OR_SENSITIVE_TERMS
        if term in searchable_brief_text
    ]

    checks.append(
        SafetyCheck(
            check_name="Protected-trait sourcing criteria",
            status="PASS" if not detected_terms else "REVIEW",
            observation=(
                "No protected or sensitive traits were detected in sourcing "
                "or screening criteria."
                if not detected_terms
                else "Detected potentially sensitive terms: "
                     + ", ".join(detected_terms)
            ),
            mitigation=(
                "The Role Intake Analyzer instruction explicitly prohibits "
                "protected personal characteristics or proxies as sourcing "
                "criteria."
            ),
        )
    )

    evidence_available = len(brief.source_evidence) > 0

    checks.append(
        SafetyCheck(
            check_name="Evidence-backed extraction",
            status="PASS" if evidence_available else "REVIEW",
            observation=(
                f"{len(brief.source_evidence)} evidence item(s) were "
                "captured for traceability."
            ),
            mitigation=(
                "Important extracted facts should retain source excerpts so "
                "recruiters can inspect where requirements came from."
            ),
        )
    )

    api_key_hardcoded = False

    checks.append(
        SafetyCheck(
            check_name="API key handling",
            status="PASS" if not api_key_hardcoded else "REVIEW",
            observation=(
                "The notebook uses Colab Secrets for the Gemini API key. "
                "The Streamlit app will use Streamlit Secrets."
            ),
            mitigation=(
                "Do not commit API keys or passwords to GitHub. Use "
                "environment variables or deployment secrets."
            ),
        )
    )

    human_approval_required = (
        brief.approval_status == ApprovalStatus.APPROVED
    )

    checks.append(
        SafetyCheck(
            check_name="Human-in-the-loop approval",
            status="PASS" if human_approval_required else "REVIEW",
            observation=(
                f"Final approval status is "
                f"{brief.approval_status.value}."
            ),
            mitigation=(
                "The agent can recommend next actions, but recruiter "
                "approval is required before sourcing launch."
            ),
        )
    )

    return checks


safety_checks = evaluate_security_and_safety(approved_brief)

print("=" * 68)
print("S.O.N.A.R. SECURITY AND SAFETY EVALUATION")
print("=" * 68)

for check in safety_checks:
    print(f"\n[{check.status}] {check.check_name}")
    print(f"Observation: {check.observation}")
    print(f"Mitigation: {check.mitigation}")

S.O.N.A.R. SECURITY AND SAFETY EVALUATION

[PASS] Protected-trait sourcing criteria
Observation: No protected or sensitive traits were detected in sourcing or screening criteria.
Mitigation: The Role Intake Analyzer instruction explicitly prohibits protected personal characteristics or proxies as sourcing criteria.

[PASS] Evidence-backed extraction
Observation: 43 evidence item(s) were captured for traceability.
Mitigation: Important extracted facts should retain source excerpts so recruiters can inspect where requirements came from.

[PASS] API key handling
Observation: The notebook uses Colab Secrets for the Gemini API key. The Streamlit app will use Streamlit Secrets.
Mitigation: Do not commit API keys or passwords to GitHub. Use environment variables or deployment secrets.

[PASS] Human-in-the-loop approval
Observation: Final approval status is approved.
Mitigation: The agent can recommend next actions, but recruiter approval is required before sourcing launch.


# 11. Final Architecture and Concepts Applied

S.O.N.A.R. is designed as a modular agentic workflow rather than a one-shot chatbot.

The system separates responsibilities across agents and tools:

```text
Recruiter inputs
JD + intake notes + clarification answers
        ↓
Role Intake Agent
Gemini + structured output
        ↓
SourcingBrief state
versioned structured memory
        ↓
Readiness Tool
deterministic scoring and gap classification
        ↓
Clarification Planner
questions generated from unresolved gaps
        ↓
Approval Router
decides next workflow action
        ↓
Human Recruiter Approval
final approval gate
        ↓
Launch Readiness Plan
rule-based sourcing launch estimate
        ↓
Evaluation Harness
signal checks, regression checks, safety checks

In [30]:
# ============================================================
# 11.1 FINAL DEMO SUMMARY OBJECT
# ============================================================

final_demo_summary = {
    "project_name": "S.O.N.A.R.",
    "full_name": "Sourcing Operations & Navigation via Agentic Recruitment",
    "track": "Agents for Business",
    "problem": (
        "Recruiting searches often begin before the role is clear enough "
        "to source consistently, causing rework, slow hiring, and poor "
        "candidate calibration."
    ),
    "solution": (
        "S.O.N.A.R. converts job descriptions, intake notes, and "
        "clarification answers into an evidence-backed sourcing brief, "
        "scores readiness using deterministic tools, routes the next "
        "action, and keeps recruiter approval in the loop."
    ),
    "models_used": {
        "brief_v1": brief_v1_model,
        "brief_v2": brief_v2_model,
        "brief_v3": brief_v3_model,
    },
    "readiness_progression": {
        "brief_v1_score": readiness_v1.total_score,
        "brief_v2_score": readiness_v2.total_score,
        "brief_v3_score": readiness_v3.total_score,
        "total_improvement": (
            readiness_v3.total_score - readiness_v1.total_score
        ),
    },
    "final_status": {
        "readiness_status": readiness_v3.readiness_status.value,
        "can_begin_sourcing": readiness_v3.can_begin_sourcing,
        "approval_status": approved_brief.approval_status.value,
        "routing_action": routing_after_approval.recommended_action.value,
    },
    "concepts_applied": [
        "agent_or_multi_agent_system",
        "tool_use",
        "structured_output",
        "agent_memory_state",
        "human_in_the_loop",
        "security_features",
        "deployability",
        "evaluation_harness",
    ],
    "evaluation_results": {
        "reliability_checks": [
            {
                "check_name": check.check_name,
                "status": check.status,
            }
            for check in reliability_checks
        ],
        "expected_signal_checks_passed": (
            f"{passed_signal_checks}/{total_signal_checks}"
        ),
        "safety_checks": [
            {
                "check_name": check.check_name,
                "status": check.status,
            }
            for check in safety_checks
        ],
    },
    "launch_plan": {
        "difficulty_band": launch_plan.difficulty_band,
        "prototype_shortlist_estimate": (
            launch_plan.prototype_shortlist_estimate
        ),
        "calibration_note": launch_plan.calibration_note,
    },
}

print("=" * 68)
print("S.O.N.A.R. FINAL HACKATHON DEMO SUMMARY")
print("=" * 68)

print(f"\nProject: {final_demo_summary['project_name']}")
print(f"Full name: {final_demo_summary['full_name']}")
print(f"Track: {final_demo_summary['track']}")

print("\nReadiness Progression")
print("---------------------")
for key, value in final_demo_summary["readiness_progression"].items():
    print(f"• {key}: {value}")

print("\nFinal Status")
print("------------")
for key, value in final_demo_summary["final_status"].items():
    print(f"• {key}: {value}")

print("\nConcepts Applied")
print("----------------")
for concept in final_demo_summary["concepts_applied"]:
    print(f"• {concept}")

print("\nEvaluation Summary")
print("------------------")
print(
    "Expected signal checks passed: "
    f"{final_demo_summary['evaluation_results']['expected_signal_checks_passed']}"
)

print("\nReliability checks:")
for check in final_demo_summary["evaluation_results"]["reliability_checks"]:
    print(f"• {check['check_name']}: {check['status']}")

print("\nSafety checks:")
for check in final_demo_summary["evaluation_results"]["safety_checks"]:
    print(f"• {check['check_name']}: {check['status']}")

print("\nLaunch Plan")
print("-----------")
for key, value in final_demo_summary["launch_plan"].items():
    print(f"• {key}: {value}")

S.O.N.A.R. FINAL HACKATHON DEMO SUMMARY

Project: S.O.N.A.R.
Full name: Sourcing Operations & Navigation via Agentic Recruitment
Track: Agents for Business

Readiness Progression
---------------------
• brief_v1_score: 28
• brief_v2_score: 88
• brief_v3_score: 93
• total_improvement: 65

Final Status
------------
• readiness_status: ready_to_source
• can_begin_sourcing: True
• approval_status: approved
• routing_action: approved_for_sourcing

Concepts Applied
----------------
• agent_or_multi_agent_system
• tool_use
• structured_output
• agent_memory_state
• human_in_the_loop
• security_features
• deployability
• evaluation_harness

Evaluation Summary
------------------
Expected signal checks passed: 9/9

Reliability checks:
• Model fallback behavior: PASS
• Screening question preservation: REVIEW
• Evidence traceability growth: PASS
• Readiness progression: PASS

Safety checks:
• Protected-trait sourcing criteria: PASS
• Evidence-backed extraction: PASS
• API key handling: PASS
• Huma